In [ ]:
# ==============================================================================
# EmotiChat v27 — TXT INTEGRADO Y ANÁLISIS EXPLICADO
# Trabajo Final: Técnicas del Procesamiento del Habla
# Tecnicatura en Ciencia de Datos e Inteligencia Artificial
# Google Colab + Python + Gradio
# ==============================================================================

# OPTIMIZACIÓN: carga diferida de modelos, caché de análisis y recursos externos concurrentes.


In [ ]:
# ==============================================================================
# 1. INSTALACIÓN DE DEPENDENCIAS
# ==============================================================================

# FFmpeg permite convertir y leer audios grabados desde distintos dispositivos.
!command -v ffmpeg >/dev/null || (apt-get -qq update && apt-get -qq install -y ffmpeg)

# EmotiChat no utiliza Weights & Biases. Se elimina para evitar el conflicto
# entre wandb y la versión de click presente en Google Colab.
!pip -q uninstall -y wandb

# Se utiliza una sola instalación para evitar reinstalaciones y conflictos.
!pip -q install --upgrade-strategy only-if-needed \
    gradio \
    transformers \
    accelerate \
    openai-whisper \
    SpeechRecognition \
    speechbrain \
    torchaudio \
    pydub \
    deep-translator \
    langdetect \
    nltk \
    textblob \
    gTTS \
    reportlab


In [ ]:
# ==============================================================================
# 2. IMPORTS


In [ ]:
# ==============================================================================

# Librerías estándar
import re
import unicodedata
import difflib
import urllib.parse
import io
import html
import json
import random
import requests
import time
import tempfile
import hashlib
import threading
from concurrent.futures import ThreadPoolExecutor
from functools import lru_cache


from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo

# Datos

# Interfaz
import gradio as gr

# Procesamiento del habla
import speech_recognition as sr
import whisper
from pydub import AudioSegment

# Traducción
from deep_translator import GoogleTranslator, MyMemoryTranslator
from langdetect import detect_langs, DetectorFactory, LangDetectException

# Resultados reproducibles en langdetect
DetectorFactory.seed = 0

# NLP
import nltk


# Sentimientos
from textblob import TextBlob
from nltk.sentiment.vader import SentimentIntensityAnalyzer

# Texto a voz
from gtts import gTTS

# Gráficos

# PDF
from reportlab.lib import colors
from reportlab.lib.enums import TA_CENTER
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle

from reportlab.platypus import (
    SimpleDocTemplate,
    Paragraph,
    Spacer,
    Table,
    TableStyle,
    Image
)

# Email

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
# Emoción en la voz
from speechbrain.inference.classifiers import EncoderClassifier


# Recursos de NLTK requeridos por VADER.
nltk.download("vader_lexicon", quiet=True)
VADER_ANALYZER = SentimentIntensityAnalyzer()


In [ ]:
# Modelo Transformers con carga diferida.
MODELO_TRANSFORMERS = "cardiffnlp/twitter-roberta-base-sentiment-latest"
tokenizer_transformers = None
modelo_transformers = None
_TRANSFORMERS_LOCK = threading.Lock()


def obtener_modelo_transformers():
    """Carga tokenizer y modelo una sola vez, durante el primer análisis."""
    global tokenizer_transformers, modelo_transformers

    if tokenizer_transformers is None or modelo_transformers is None:
        with _TRANSFORMERS_LOCK:
            if tokenizer_transformers is None or modelo_transformers is None:
                print("Cargando modelo Transformers por primera vez...")
                tokenizer_transformers = AutoTokenizer.from_pretrained(MODELO_TRANSFORMERS)
                modelo_transformers = AutoModelForSequenceClassification.from_pretrained(
                    MODELO_TRANSFORMERS
                )
                modelo_transformers.eval()
                print("Modelo Transformers cargado correctamente.")

    return tokenizer_transformers, modelo_transformers


In [ ]:
# ==============================================================================
# 3. CONFIGURACIÓN GENERAL


In [ ]:
# ==============================================================================

APP_NAME = "EmotiChat"

BASE_DIR = Path("emotichat")

AUDIO_DIR = BASE_DIR / "audio"

AUDIO_DIR.mkdir(parents=True, exist_ok=True)

LOGO_URL = "https://raw.githubusercontent.com/PaoRioColorado/EmotiChat/main/Logo_EmotiChat.png"
IDIOMAS = {
    "en": "Inglés",
    "fr": "Francés",
    "de": "Alemán",
    "pt": "Portugués"
}


# ==============================================================================
# CONTENIDO CONVERSACIONAL EXTERNO
# ==============================================================================

EMOTICHAT_DATA_URL = (
    "https://raw.githubusercontent.com/"
    "PaoRioColorado/EmotiChat/main/data/emotichat_data.json"
)

RUTA_DATOS_LOCAL = Path("/content/emotichat_data.json")

def cargar_datos_conversacionales(forzar_descarga=False):
    """
    1. Usa el JSON local si ya existe.
    2. Si no existe, lo descarga desde GitHub.
    3. Guarda una copia en /content.
    """

    if RUTA_DATOS_LOCAL.exists() and not forzar_descarga:
        try:
            with RUTA_DATOS_LOCAL.open("r", encoding="utf-8") as archivo:
                datos = json.load(archivo)

            print("Datos conversacionales cargados desde Colab.")
            return datos

        except (json.JSONDecodeError, OSError) as error_local:
            print(f"La copia local no pudo leerse: {error_local}")
            print("Se intentará descargar nuevamente desde GitHub.")

    try:
        respuesta = requests.get(EMOTICHAT_DATA_URL, timeout=10)
        respuesta.raise_for_status()
        datos = respuesta.json()

        RUTA_DATOS_LOCAL.write_text(
            json.dumps(datos, ensure_ascii=False, indent=2),
            encoding="utf-8"
        )

        print("Datos conversacionales descargados desde GitHub.")
        return datos

    except requests.RequestException as error_red:
        raise RuntimeError(
            "No se pudo descargar emotichat_data.json desde GitHub. "
            "Revisá la conexión o subí el archivo manualmente a "
            "/content/emotichat_data.json. "
            f"Detalle: {error_red}"
        ) from error_red

    except json.JSONDecodeError as error_json:
        raise RuntimeError(
            "El archivo descargado no contiene un JSON válido. "
            f"Detalle: {error_json}"
        ) from error_json

DATOS_CONVERSACION = cargar_datos_conversacionales()

NOMBRES_IDIOMAS = {
    "es": "Español",
    "en": "Inglés",
    "fr": "Francés",
    "de": "Alemán",
    "pt": "Portugués"
}



# Configuración de transcripción
WHISPER_MODEL_NAME = "base"
WHISPER_MODEL = None

# Modelo experimental para reconocimiento de emociones en la voz.
SPEECHBRAIN_MODEL = None
SPEECHBRAIN_MODEL_NAME = "speechbrain/emotion-recognition-wav2vec2-IEMOCAP"


In [ ]:
# ==============================================================================
# 4. RECURSOS NLTK


In [ ]:
# No se descargan corpus adicionales de NLTK.
# EmotiChat utiliza únicamente vader_lexicon, cargado en la celda de imports.
print("Recursos de NLTK listos.")


In [ ]:
# ==============================================================================
# 5. MOTOR DE PROCESAMIENTO Y MODELOS


In [ ]:
# ==============================================================================

class EmotiChatEngine:
    def detectar_idioma_detallado(self, texto):
        """
        Detecta el idioma y devuelve también el nivel de confianza visible.

        langdetect puede equivocarse con mensajes muy breves. Para evitar
        falsos positivos, EmotiChat reconoce marcadores frecuentes del español
        y utiliza "Estimado" cuando el texto no aporta suficiente contexto.
        """
        texto = (texto or "").strip()

        if not texto:
            return "es", "Estimado"

        palabras = re.findall(
            r"[a-záéíóúüñ]+",
            texto.lower(),
            flags=re.IGNORECASE
        )

        marcadores_es = {
            "a", "al", "algo", "ando", "como", "con", "cuando", "de", "del",
            "el", "ella", "en", "es", "esta", "está", "estaba", "estoy",
            "hoy", "hola", "la", "las", "le", "lo", "los", "me", "mi",
            "mis", "muy", "no", "nos", "para", "pero", "porque", "por",
            "que", "qué", "se", "siento", "sin", "soy", "su", "te",
            "tengo", "tu", "un", "una", "y", "ya", "yo",
            "bien", "mal", "feliz", "triste", "ansiosa", "ansioso",
            "cansada", "cansado", "preocupada", "preocupado",
            "enojada", "enojado", "novio", "novia", "bautiza",
            "resfriada", "resfriado", "enferma", "enfermo", "dolor",
            "garganta", "cabeza", "panza", "trabajo", "familia"
        }

        coincidencias_es = sum(
            1 for palabra in palabras if palabra in marcadores_es
        )

        # Los mensajes breves no permiten una detección estadística estable.
        # Si contienen vocabulario español, se informa como estimación.
        if len(palabras) < 5 and coincidencias_es >= 1:
            return "es", "Estimado"

        if len(texto) < 25 and coincidencias_es >= 2:
            return "es", "Estimado"

        try:
            resultados = detect_langs(texto)

            if not resultados:
                return "es", "Estimado"

            mejor = resultados[0]

            # Una probabilidad baja no se presenta como una certeza.
            if mejor.prob < 0.80:
                if coincidencias_es >= 1:
                    return "es", "Estimado"
                return mejor.lang, "Estimado"

            return mejor.lang, "Confiable"

        except LangDetectException:
            return "es", "Estimado"
        except Exception:
            return "es", "Estimado"

    def traducir(self, texto, origen, destino):
        """Traduce con Google y utiliza MyMemory si el primer servicio falla."""
        texto = str(texto or "").strip()
        if not texto:
            return ""
        if origen == destino:
            return texto

        nombres_mymemory = {
            "auto": "auto",
            "es": "spanish",
            "en": "english",
            "fr": "french",
            "de": "german",
            "pt": "portuguese",
        }
        proveedores = (
            lambda: GoogleTranslator(source=origen, target=destino).translate(texto),
            lambda: MyMemoryTranslator(
                source=nombres_mymemory.get(origen, origen),
                target=nombres_mymemory.get(destino, destino)
            ).translate(texto),
        )
        for traducir_con_proveedor in proveedores:
            try:
                resultado = str(traducir_con_proveedor() or "").strip()
                if resultado:
                    return resultado
            except Exception:
                continue
        return "Traducción no disponible temporalmente."

    def traduccion_encadenada(self, texto):
        """Traduce cada idioma desde el original para evitar errores acumulados."""
        idioma_origen, confianza_idioma = self.detectar_idioma_detallado(texto)
        pasos = []
        traduccion_ingles = ""

        for codigo, nombre in IDIOMAS.items():
            traducido = self.traducir(texto, idioma_origen, codigo)
            pasos.append({"idioma": nombre, "texto": traducido})
            if codigo == "en":
                traduccion_ingles = traducido

        disponible = bool(
            traduccion_ingles
            and not traduccion_ingles.startswith("Traducción no disponible")
        )
        final = (
            self.traducir(traduccion_ingles, "en", "es")
            if disponible
            else "Traducción no disponible temporalmente."
        )

        return {
            "idioma_origen": idioma_origen,
            "confianza_idioma": confianza_idioma,
            "original": texto,
            "pasos": pasos,
            "final": final,
            "disponible": disponible and not final.startswith("Traducción no disponible")
        }

    @lru_cache(maxsize=256)
    def sentimiento_textblob(self, texto):
        """Analiza directamente el texto recibido, sin traducirlo internamente."""
        try:
            texto = str(texto or "").strip()
            if not texto:
                raise ValueError("El texto está vacío.")
            blob = TextBlob(texto)
            polaridad = float(blob.sentiment.polarity)
            subjetividad = float(blob.sentiment.subjectivity)
            sentimiento = "Positivo" if polaridad > 0.15 else "Negativo" if polaridad < -0.15 else "Neutro"
            return {
                "sentimiento": sentimiento,
                "polaridad": round(polaridad, 3),
                "subjetividad": round(subjetividad, 3)
            }
        except Exception as error:
            return {"sentimiento": "No disponible", "polaridad": 0, "subjetividad": 0, "error": str(error)}

    @lru_cache(maxsize=256)
    def sentimiento_vader(self, texto):
        """Aplica VADER de NLTK directamente al texto recibido."""
        try:
            texto = str(texto or "").strip()
            if not texto:
                raise ValueError("El texto está vacío.")
            scores = VADER_ANALYZER.polarity_scores(texto)
            compound = float(scores["compound"])
            sentimiento = "Positivo" if compound >= 0.05 else "Negativo" if compound <= -0.05 else "Neutro"
            return {
                "sentimiento": sentimiento,
                "compound": round(compound, 3),
                "positivo": round(float(scores["pos"]), 3),
                "neutro": round(float(scores["neu"]), 3),
                "negativo": round(float(scores["neg"]), 3)
            }
        except Exception as error:
            return {"sentimiento": "No disponible", "compound": 0, "error": str(error)}

    def normalizar_texto(self, texto):
        texto = str(texto or "").lower().strip()
        texto = unicodedata.normalize("NFD", texto)
        texto = "".join(
            caracter for caracter in texto
            if unicodedata.category(caracter) != "Mn"
        )
        texto = re.sub(r"\s+", " ", texto)
        return texto

    def calcular_wer(self, referencia, hipotesis):
        """Calcula Word Error Rate mediante distancia de Levenshtein por palabras."""
        palabras_ref = self.normalizar_texto(referencia).split()
        palabras_hip = self.normalizar_texto(hipotesis).split()
        if not palabras_ref:
            return 0.0 if not palabras_hip else 100.0

        anterior = list(range(len(palabras_hip) + 1))
        for indice_ref, palabra_ref in enumerate(palabras_ref, start=1):
            actual = [indice_ref]
            for indice_hip, palabra_hip in enumerate(palabras_hip, start=1):
                sustitucion = anterior[indice_hip - 1] + (palabra_ref != palabra_hip)
                insercion = actual[indice_hip - 1] + 1
                eliminacion = anterior[indice_hip] + 1
                actual.append(min(sustitucion, insercion, eliminacion))
            anterior = actual

        return round((anterior[-1] / len(palabras_ref)) * 100, 1)

    def detectar_negacion_emocional(self, texto):
        """
        Detecta expresiones negativas explícitas que pueden confundirse
        por contener palabras como 'bien'.
        """
        texto_limpio = self.normalizar_texto(texto)

        frases_negativas = [
            "no me siento bien",
            "no estoy bien",
            "no ando bien",
            "no me encuentro bien",
            "no me siento nada bien",
            "no estoy nada bien",
            "para nada bien",
            "me siento mal",
            "estoy mal",
            "me encuentro mal",
            "no puedo mas",
            "estoy angustiado",
            "estoy angustiada",
            "estoy triste",
            "estoy preocupado",
            "estoy preocupada",
            "me duele",
            "tengo dolor",
            "estoy cansado",
            "estoy cansada"
        ]

        return any(frase in texto_limpio for frase in frases_negativas)

    def ajustar_sentimiento_contextual(self, texto, sentimiento, confianza):
        """
        Corrige casos claros de negación en español.
        El modelo sigue siendo el método principal; las reglas solamente
        evitan contradicciones evidentes como 'no me siento bien'.
        """
        if self.detectar_negacion_emocional(texto):
            return {
                "sentimiento": "Negativo",
                "confianza": max(float(confianza), 0.90),
                "ajuste_contextual": True
            }

        return {
            "sentimiento": sentimiento,
            "confianza": float(confianza),
            "ajuste_contextual": False
        }

    @lru_cache(maxsize=256)
    def sentimiento_transformers(self, texto):
        """Aplica el modelo Transformers directamente al texto recibido."""
        try:
            texto = str(texto or "").strip()
            if not texto:
                raise ValueError("El texto está vacío.")
            tokenizer, modelo = obtener_modelo_transformers()
            entradas = tokenizer(texto, return_tensors="pt", truncation=True, max_length=512)
            with torch.inference_mode():
                salida = modelo(**entradas)
            probabilidades = torch.softmax(salida.logits, dim=-1)[0]
            indice = int(torch.argmax(probabilidades).item())
            confianza = float(probabilidades[indice].item())
            etiqueta = str(modelo.config.id2label[indice]).lower()
            if "positive" in etiqueta or etiqueta in {"2", "label_2"}:
                sentimiento = "Positivo"
            elif "negative" in etiqueta or etiqueta in {"0", "label_0"}:
                sentimiento = "Negativo"
            else:
                sentimiento = "Neutro"
            ajuste = self.ajustar_sentimiento_contextual(texto, sentimiento, confianza)
            return {
                "sentimiento": ajuste["sentimiento"],
                "confianza": round(ajuste["confianza"], 3),
                "etiqueta_modelo": etiqueta,
                "ajuste_contextual": ajuste["ajuste_contextual"]
            }
        except Exception as error:
            return {"sentimiento": "No disponible", "confianza": 0, "error": str(error)}

    def obtener_modelo_whisper(self):
        """Carga Whisper base una sola vez y reutiliza el modelo en toda la aplicación."""
        global WHISPER_MODEL

        if WHISPER_MODEL is None:
            print("Cargando Whisper base...")
            WHISPER_MODEL = whisper.load_model(WHISPER_MODEL_NAME)
        return WHISPER_MODEL

    def transcribir_whisper(self, ruta_audio):
        """
        Transcribe un archivo de audio con Whisper y devuelve información
        útil para mostrar en la interfaz.
        """
        if not ruta_audio:
            return {
                "ok": False,
                "texto": "",
                "idioma": "—",
                "tiempo": 0,
                "error": "No se recibió ningún archivo de audio."
            }

        inicio = time.time()

        try:
            modelo = self.obtener_modelo_whisper()
            resultado = modelo.transcribe(
                str(ruta_audio),
                fp16=False,
                language="es",
                task="transcribe",
                temperature=0.0,
                condition_on_previous_text=False,
                initial_prompt=None
            )

            texto = (resultado.get("text") or "").strip()
            idioma_codigo = resultado.get("language", "—")
            idioma = NOMBRES_IDIOMAS.get(
                idioma_codigo,
                str(idioma_codigo).upper()
            )

            if not texto:
                return {
                    "ok": False,
                    "texto": "",
                    "idioma": idioma,
                    "tiempo": round(time.time() - inicio, 2),
                    "error": "Whisper no pudo reconocer palabras en el audio."
                }

            return {
                "ok": True,
                "texto": texto,
                "idioma": idioma,
                "tiempo": round(time.time() - inicio, 2),
                "error": ""
            }

        except Exception as error:
            return {
                "ok": False,
                "texto": "",
                "idioma": "—",
                "tiempo": round(time.time() - inicio, 2),
                "error": f"No se pudo transcribir el audio: {error}"
            }

    def preparar_audio_wav(self, ruta_audio):
        """
        Convierte el audio recibido a WAV mono de 16 kHz.
        Esto permite que SpeechRecognition procese también archivos
        grabados o subidos en otros formatos.
        """
        if not ruta_audio:
            raise ValueError("No se recibió ningún archivo de audio.")

        ruta_audio = Path(ruta_audio)
        ruta_wav = AUDIO_DIR / f"audio_convertido_{int(time.time() * 1000)}.wav"

        audio = AudioSegment.from_file(str(ruta_audio))
        audio = audio.set_channels(1).set_frame_rate(16000)
        audio.export(str(ruta_wav), format="wav")

        return str(ruta_wav)

    def transcribir_speech_recognition(self, ruta_audio):
        """
        Transcribe audio con SpeechRecognition usando Google Web Speech.
        Devuelve texto, tiempo y posibles errores.
        """
        inicio = time.time()
        ruta_wav = None

        try:
            ruta_wav = self.preparar_audio_wav(ruta_audio)
            reconocedor = sr.Recognizer()

            with sr.AudioFile(ruta_wav) as fuente:
                audio = reconocedor.record(fuente)

            texto = reconocedor.recognize_google(
                audio,
                language="es-AR"
            ).strip()

            return {
                "ok": True,
                "texto": texto,
                "idioma": "Español",
                "tiempo": round(time.time() - inicio, 2),
                "error": ""
            }

        except sr.UnknownValueError:
            return {
                "ok": False,
                "texto": "",
                "idioma": "Español",
                "tiempo": round(time.time() - inicio, 2),
                "error": "SpeechRecognition no pudo comprender el audio."
            }

        except sr.RequestError as error:
            return {
                "ok": False,
                "texto": "",
                "idioma": "Español",
                "tiempo": round(time.time() - inicio, 2),
                "error": f"El servicio de reconocimiento no respondió: {error}"
            }

        except Exception as error:
            return {
                "ok": False,
                "texto": "",
                "idioma": "—",
                "tiempo": round(time.time() - inicio, 2),
                "error": f"No se pudo procesar el audio: {error}"
            }

        finally:
            if ruta_wav:
                try:
                    Path(ruta_wav).unlink(missing_ok=True)
                except Exception:
                    pass

    def comparar_transcripciones(self, ruta_audio):
        """
        Ejecuta Whisper y SpeechRecognition sobre el mismo audio.
        Whisper se usa como texto principal del chat porque admite
        detección automática de idioma y suele ser más robusto.
        """
        whisper_resultado = self.transcribir_whisper(ruta_audio)
        speech_resultado = self.transcribir_speech_recognition(ruta_audio)

        textos_iguales = False

        if whisper_resultado.get("ok") and speech_resultado.get("ok"):
            normalizar = lambda texto: re.sub(
                r"[^a-záéíóúüñ0-9 ]+",
                "",
                texto.lower()
            ).strip()

            textos_iguales = (
                normalizar(whisper_resultado["texto"]) ==
                normalizar(speech_resultado["texto"])
            )

        if whisper_resultado.get("ok") and speech_resultado.get("ok"):
            conclusion = (
                "Ambos métodos obtuvieron una transcripción equivalente."
                if textos_iguales
                else
                "Los métodos presentan diferencias. Whisper se usa como "
                "resultado principal por su detección automática de idioma."
            )
        elif whisper_resultado.get("ok"):
            conclusion = (
                "Whisper completó la transcripción, pero SpeechRecognition "
                "no pudo procesar correctamente el audio."
            )
        elif speech_resultado.get("ok"):
            conclusion = (
                "SpeechRecognition completó la transcripción, mientras que "
                "Whisper no pudo procesar correctamente el audio."
            )
        else:
            conclusion = (
                "Ninguno de los métodos pudo obtener una transcripción válida."
            )

        return {
            "whisper": whisper_resultado,
            "speech_recognition": speech_resultado,
            "coinciden": textos_iguales,
            "conclusion": conclusion
        }

    def comparar_sentimiento_original_traduccion(self, texto):
        """Compara los tres métodos en español y en la traducción inglesa."""
        try:
            texto_es = str(texto or "").strip()
            if not texto_es:
                raise ValueError("No hay texto para analizar.")
            texto_en = self.traducir(texto_es, "auto", "en")
            if not texto_en or str(texto_en).startswith("Traducción no disponible"):
                raise RuntimeError(str(texto_en))

            resultados = {
                "TextBlob": {
                    "es": self.sentimiento_textblob(texto_es),
                    "en": self.sentimiento_textblob(texto_en),
                    "metrica": "polaridad"
                },
                "VADER (NLTK)": {
                    "es": self.sentimiento_vader(texto_es),
                    "en": self.sentimiento_vader(texto_en),
                    "metrica": "compound"
                },
                "Transformers": {
                    "es": self.sentimiento_transformers(texto_es),
                    "en": self.sentimiento_transformers(texto_en),
                    "metrica": "confianza"
                }
            }

            for metodo, datos in resultados.items():
                es, en = datos["es"], datos["en"]
                mismo = es.get("sentimiento") == en.get("sentimiento")
                metrica = datos["metrica"]
                diferencia_valor = abs(float(es.get(metrica, 0)) - float(en.get(metrica, 0)))
                datos["coinciden"] = mismo
                datos["diferencia_valor"] = round(diferencia_valor, 3)
                datos["analisis"] = (
                    "Coinciden en la categoría; cambia la intensidad."
                    if mismo and diferencia_valor >= 0.15 else
                    "Coinciden en categoría e intensidad aproximada."
                    if mismo else
                    "La traducción modifica la categoría detectada."
                )

            return {
                "texto_original": texto_es,
                "texto_ingles": texto_en,
                "resultados": resultados,
                "error": ""
            }
        except Exception as error:
            return {"texto_original": str(texto or ""), "texto_ingles": "", "resultados": {}, "error": str(error)}

    def descargar_audios_prueba(self):
        """Descarga y valida los dos WAV obligatorios desde Google Drive.

        Si la descarga desde Drive falla, utiliza GitHub como respaldo.
        Los componentes de Gradio reciben rutas locales, no enlaces remotos.
        """
        fuentes = {
            "Informativo": [
                "https://drive.google.com/uc?export=download&id=1s7Djpz6v8Qo1a6b-vR5CFcnQZRZVsOd1",
                "https://raw.githubusercontent.com/PaoRioColorado/EmotiChat/main/Audio%20informativo.wav",
            ],
            "Emocional": [
                "https://drive.google.com/uc?export=download&id=1gOuioVZv2-KfgQCPj2fTcmQ9VcqK2FSD",
                "https://raw.githubusercontent.com/PaoRioColorado/EmotiChat/main/Audio%20emocional.wav",
            ],
        }

        def es_wav_valido(ruta):
            try:
                import wave
                with wave.open(str(ruta), "rb") as audio_wav:
                    return audio_wav.getnframes() > 0 and audio_wav.getframerate() > 0
            except Exception:
                return False

        rutas = {}
        AUDIO_DIR.mkdir(parents=True, exist_ok=True)

        for nombre, urls in fuentes.items():
            destino = AUDIO_DIR / f"audio_{nombre.lower()}.wav"

            if destino.exists() and es_wav_valido(destino):
                rutas[nombre] = str(destino)
                continue

            ultimo_error = None
            for url in urls:
                try:
                    respuesta = requests.get(
                        url,
                        timeout=20,
                        allow_redirects=True,
                        headers={"User-Agent": "Mozilla/5.0"},
                    )
                    respuesta.raise_for_status()

                    contenido = respuesta.content
                    tipo = respuesta.headers.get("content-type", "").lower()
                    if len(contenido) < 1000 or b"<html" in contenido[:500].lower():
                        raise RuntimeError(
                            f"La dirección devolvió una página web en lugar del WAV ({tipo or 'sin tipo'})."
                        )

                    destino.write_bytes(contenido)
                    if not es_wav_valido(destino):
                        destino.unlink(missing_ok=True)
                        raise RuntimeError("El archivo descargado no es un WAV válido.")

                    rutas[nombre] = str(destino)
                    break
                except Exception as error:
                    ultimo_error = error

            if nombre not in rutas:
                raise RuntimeError(
                    f"No se pudo descargar el audio {nombre.lower()}: {ultimo_error}"
                )

        return rutas

    def comparar_audios_obligatorios(self):
        """Procesa los WAV y calcula WER y precisión estimada."""
        referencias = {
            "Informativo": (
                "Hola. Este es un audio de prueba para el proyecto EmotiChat. "
                "Hoy es un día tranquilo. Durante la mañana trabajé, respondí algunos correos "
                "y luego continué con mis actividades habituales. Este mensaje tiene un tono informativo."
            ),
            "Emocional": (
                "Hola, estoy muy preocupada porque mi perrito se perdió esta mañana y todavía no aparece. "
                "Lo busqué por varias calles y me siento muy angustiada. Espero de verdad que alguien lo "
                "encuentre y pueda volver sano y salvo a casa."
            )
        }
        rutas = self.descargar_audios_prueba()
        filas = []
        for tipo, ruta in rutas.items():
            comparacion = self.comparar_transcripciones(ruta)
            referencia = referencias[tipo]
            for metodo, clave in [("SpeechRecognition", "speech_recognition"), ("Whisper", "whisper")]:
                resultado = comparacion[clave]
                texto = resultado.get("texto", "")
                wer = self.calcular_wer(referencia, texto) if texto else 100.0
                similitud = max(0.0, 100.0 - wer)
                filas.append({
                    "audio": tipo,
                    "metodo": metodo,
                    "transcripcion": texto or resultado.get("error", "Sin resultado"),
                    "tiempo": float(resultado.get("tiempo", 0)),
                    "precision": round(similitud, 1),
                    "wer": round(wer, 1),
                    "observacion": (
                        "Alta precisión" if similitud >= 85 else
                        "Precisión media; revisar acento, velocidad o puntuación" if similitud >= 60 else
                        "Baja precisión; posible efecto de ruido, acento o velocidad"
                    )
                })
        return filas

    def generar_resumen_pipeline(self, texto):
        """
        Genera un resumen breve del análisis para completar el pipeline:
        Audio → Transcripción → Sentimiento → Resumen → Nuevo audio.
        """
        tb = self.sentimiento_textblob(texto)
        vd = self.sentimiento_vader(texto)
        tr = self.sentimiento_transformers(texto)

        resultados = [
            tb.get("sentimiento", "No disponible"),
            vd.get("sentimiento", "No disponible"),
            tr.get("sentimiento", "No disponible")
        ]
        validos = [r for r in resultados if r != "No disponible"]

        if validos:
            sentimiento_general = max(set(validos), key=validos.count)
        else:
            sentimiento_general = "No disponible"

        resumen = (
            f"El sentimiento predominante del mensaje fue "
            f"{sentimiento_general.lower()}. "
            f"TextBlob indicó {tb.get('sentimiento', 'no disponible')}, "
            f"VADER indicó {vd.get('sentimiento', 'no disponible')} y "
            f"Transformers indicó {tr.get('sentimiento', 'no disponible')}. "
            "Los resultados son orientativos y pueden verse afectados por "
            "ironía, contexto, traducción, acento o ambigüedad."
        )

        return {
            "texto": texto,
            "sentimiento_general": sentimiento_general,
            "textblob": tb,
            "vader": vd,
            "transformers": tr,
            "resumen": resumen
        }

    def obtener_modelo_emocion_voz(self):
        """
        Carga SpeechBrain una sola vez y conserva el modelo en memoria.
        Se utiliza como funcionalidad experimental adicional.
        """
        global SPEECHBRAIN_MODEL

        if SPEECHBRAIN_MODEL is None:
            print("Cargando modelo SpeechBrain de emociones en la voz...")

            SPEECHBRAIN_MODEL = EncoderClassifier.from_hparams(
                source=SPEECHBRAIN_MODEL_NAME,
                savedir="pretrained_models/emotion"
            )

            print("Modelo SpeechBrain cargado correctamente.")

        return SPEECHBRAIN_MODEL

    def analizar_emocion_voz(self, ruta_audio):
        """
        Estima una emoción acústica a partir del audio.
        El modelo devuelve categorías IEMOCAP:
        ang, hap, sad y neu.
        """
        if not ruta_audio:
            return {
                "ok": False,
                "emocion": "No disponible",
                "confianza": 0,
                "etiqueta": "",
                "error": "No se recibió audio."
            }

        inicio = time.perf_counter()

        etiquetas = {
            "ang": "Enojo",
            "hap": "Alegría",
            "sad": "Tristeza",
            "neu": "Neutral"
        }

        try:
            modelo = self.obtener_modelo_emocion_voz()
            salida = modelo.classify_file(str(ruta_audio))

            _, score, _, text_lab = salida

            if isinstance(text_lab, (list, tuple)):
                etiqueta = str(text_lab[0]).lower()
            else:
                etiqueta = str(text_lab).lower()

            etiqueta = etiqueta.replace("[", "").replace("]", "").replace("'", "").strip()
            confianza = float(score.squeeze().detach().cpu().item())

            return {
                "ok": True,
                "emocion": etiquetas.get(etiqueta, etiqueta.title()),
                "confianza": round(confianza, 3),
                "etiqueta": etiqueta,
                "tiempo": round(time.perf_counter() - inicio, 2),
                "error": ""
            }

        except Exception as error:
            return {
                "ok": False,
                "emocion": "No disponible",
                "confianza": 0,
                "etiqueta": "",
                "tiempo": round(time.perf_counter() - inicio, 2),
                "error": str(error)
            }

    def comparar_texto_y_voz(self, texto, resultado_voz):
        """
        Compara el sentimiento del contenido textual con la emoción acústica.
        """
        resultado_texto = self.sentimiento_transformers(texto)
        sentimiento_texto = resultado_texto.get("sentimiento", "No disponible")
        emocion_voz = resultado_voz.get("emocion", "No disponible")

        compatibles = {
            "Positivo": {"Alegría"},
            "Negativo": {"Tristeza", "Enojo"},
            "Neutro": {"Neutral"}
        }

        coincide = (
            resultado_voz.get("ok", False)
            and emocion_voz in compatibles.get(sentimiento_texto, set())
        )

        if not resultado_voz.get("ok"):
            conclusion = (
                "No fue posible estimar la emoción acústica del audio."
            )
        elif coincide:
            conclusion = (
                "El contenido del mensaje y la emoción detectada en la voz "
                "presentan una concordancia general."
            )
        else:
            conclusion = (
                "Se detectó una posible discrepancia entre lo que expresa el "
                "texto y cómo suena la voz. Este resultado es experimental."
            )

        return {
            "sentimiento_texto": sentimiento_texto,
            "confianza_texto": resultado_texto.get("confianza", 0),
            "emocion_voz": emocion_voz,
            "confianza_voz": resultado_voz.get("confianza", 0),
            "coincide": coincide,
            "conclusion": conclusion
        }

    def texto_a_voz_configurable(self, texto, idioma="es", lento=False):
        try:
            ruta = AUDIO_DIR / (
                f"tts_{idioma}_{'lento' if lento else 'normal'}_"
                f"{int(time.time() * 1000)}.mp3"
            )
            gTTS(text=texto, lang=idioma, slow=lento).save(str(ruta))
            return str(ruta)
        except Exception:
            return None

    def texto_a_voz(self, texto):
        try:
            ruta = AUDIO_DIR / f"respuesta_{int(time.time())}.mp3"
            gTTS(text=texto, lang="es").save(str(ruta))
            return str(ruta)
        except Exception:
            return None

    def detectar_crisis(self, texto):
        limpio = self.normalizar_texto(texto)
        frases = [
            "me quiero matar", "quiero matarme", "me mato",
            "me quiero morir", "quiero morirme", "no quiero vivir",
            "me quito la vida", "quitarme la vida",
            "terminar con mi vida", "acabar con mi vida",
            "quiero desaparecer", "seria mejor no estar",
            "me voy a hacer daño", "quiero hacerme daño",
            "estoy cansado de vivir", "estoy cansada de vivir"
        ]
        if any(frase in limpio for frase in frases):
            return True

        claves = ["matar", "matarme", "morir", "morirme", "suicidio", "suicidarme"]
        for token in limpio.split():
            similitud = max(
                difflib.SequenceMatcher(None, token, clave).ratio()
                for clave in claves
            )
            if similitud >= 0.84 and any(
                indicio in limpio for indicio in ["quiero", "me ", "vida", "no puedo"]
            ):
                return True
        return False

    def tarjeta_crisis_html(self, nombre):
        mensaje = (
            "Hola. No me estoy sintiendo bien y necesito que alguien "
            "esté conmigo o me llame ahora. ¿Podés ayudarme?"
        )
        mensaje_url = urllib.parse.quote(mensaje)

        return f"""
        <div class="crisis-card">
            <div class="crisis-heading">🚨 Apoyo inmediato</div>
            <p><strong>{html.escape(nombre)}, lamento mucho que estés pasando por esto.</strong></p>
            <p>¿Estás en peligro inmediato o pensás hacerte daño ahora?</p>
            <p>
                Si la respuesta es sí, llamá a emergencias o pedile a alguien
                cercano que se quede con vos.
            </p>

            <div class="crisis-actions">
                <a class="crisis-btn whatsapp"
                   href="https://wa.me/?text={mensaje_url}"
                   target="_blank" rel="noopener">Chat Abrir WhatsApp</a>

                <button class="crisis-btn copy-message"
                        onclick="navigator.clipboard.writeText(`{mensaje}`)">
                   📋 Copiar mensaje
                </button>

                <a class="crisis-btn emergency" href="tel:911">📞 Llamar al 911</a>
                <a class="crisis-btn emergency-secondary" href="tel:08009990091">
                   🧠 Salud mental 24 h
                </a>
            </div>

            <details class="crisis-details">
                <summary>Ver otras líneas de ayuda</summary>
                <p><a href="tel:135">Línea 135</a></p>
                <p><a href="tel:107">Emergencias médicas 107</a></p>
            </details>

            <div class="ethical-note">
                EmotiChat no reemplaza la atención profesional.
            </div>
        </div>
        """

    def obtener_recursos_contextuales(self, texto):
        limpio = self.normalizar_texto(texto)

        if any(p in limpio for p in ["nervios", "nerviosa", "nervioso", "ansiedad", "ansiosa", "ansioso"]):
            return [
                ("🫁", "Respiración guiada", "Inhalá 4 segundos, sostené 2 y exhalá 6."),
                ("🧭", "Técnica 5-4-3-2-1", "Nombrá 5 cosas que ves, 4 que tocás, 3 que oís, 2 que olés y 1 que saboreás."),
                ("🤝", "Apoyo cercano", "Contale a alguien de confianza que estás pasando un momento difícil.")
            ]

        if any(p in limpio for p in ["estres", "agotada", "agotado", "cansada", "cansado"]):
            return [
                ("⏸️", "Pausa breve", "Alejate cinco minutos de la pantalla."),
                ("💧", "Hidratación", "Tomá agua y revisá si comiste y descansaste."),
                ("🧘", "Relajación muscular", "Tensá y soltá lentamente manos, hombros y piernas.")
            ]

        if any(p in limpio for p in ["triste", "mal", "angustiada", "angustiado", "sola", "solo"]):
            return [
                ("📝", "Escribir lo que sentís", "Anotá en una frase qué te preocupa más ahora."),
                ("🤝", "Hablar con alguien", "Elegí una persona de confianza y contale cómo te sentís."),
                ("🌿", "Respiración lenta", "Respirá suavemente durante uno o dos minutos.")
            ]

        if any(p in limpio for p in ["no puedo dormir", "insomnio", "dormir"]):
            return [
                ("🌙", "Preparar el descanso", "Bajá luces y evitá pantallas unos minutos."),
                ("🫁", "Respiración 4-6", "Inhalá 4 segundos y exhalá 6 durante cinco ciclos."),
                ("📝", "Descargar pensamientos", "Anotá lo pendiente para retomarlo mañana.")
            ]
        return []

    def recursos_html(self, texto):
        recursos = self.obtener_recursos_contextuales(texto)
        if not recursos:
            return ""

        tarjetas = "".join(
            f"""
            <div class="resource-item">
                <span class="resource-icon">{icono}</span>
                <div>
                    <strong>{html.escape(titulo)}</strong>
                    <p>{html.escape(descripcion)}</p>
                </div>
            </div>
            """
            for icono, titulo, descripcion in recursos
        )

        return f"""
        <div class="resources-card">
            <h3>🌿 Recursos recomendados</h3>
            {tarjetas}
            <p class="resource-note">
                Son sugerencias generales y no reemplazan la orientación profesional.
            </p>
        </div>
        """

    def contiene_expresion(self, texto, expresiones):
        """
        Busca palabras o frases completas para evitar coincidencias falsas,
        por ejemplo 'mal' dentro de 'normal'.
        """
        texto_limpio = self.normalizar_texto(texto)

        for expresion in expresiones:
            expresion_limpia = self.normalizar_texto(expresion)
            patron = rf"(?<!\w){re.escape(expresion_limpia)}(?!\w)"

            if re.search(patron, texto_limpio):
                return True

        return False

    def elegir_respuesta(self, categoria, nombre):
        opciones = DATOS_CONVERSACION["respuestas"].get(
            categoria,
            DATOS_CONVERSACION["respuestas"]["general"]
        )

        plantilla = random.choice(opciones)
        if not nombre or nombre == "Invitad@":
            plantilla = re.sub(
                r"\s*\{nombre\}\s*[,;:]?\s*",
                " ",
                plantilla
            )
            respuesta = plantilla.format(nombre="")
            respuesta = re.sub(r"\s+([,.!?])", r"\1", respuesta)
            return re.sub(r"\s{2,}", " ", respuesta).strip()

        return plantilla.format(nombre=nombre)

    def responder(self, texto, nombre):
        """
        Respuesta conversacional mediante un único archivo JSON externo.
        La seguridad y las consultas físicas conservan prioridad.
        """
        original = str(texto or "").strip()
        limpio = self.normalizar_texto(original)

        # La detección de crisis siempre se procesa primero.
        if self.detectar_crisis(original):
            return self.tarjeta_crisis_html(nombre)

        # Orientación prudente ante dolor o malestar físico.
        if any(frase in limpio for frase in [
            "dolor de cabeza",
            "me duele la cabeza",
            "tengo migraña",
            "tengo migrana"
        ]):
            return (
                f"{nombre}, lamento que te duela la cabeza. "
                "Podés probar descansar en un lugar tranquilo, hidratarte y "
                "reducir pantallas o luces intensas. Si el dolor es repentino "
                "y muy fuerte, aparece con fiebre alta, confusión, debilidad, "
                "dificultad para hablar o no mejora, buscá atención médica."
            )

        # Si la persona ya expresó que se siente bien, la respuesta reconoce
        # esa información y evita volver a preguntarle cómo se siente.
        expresiones_positivas_directas = [
            "me siento bien", "me siento muy bien", "estoy bien",
            "estoy muy bien", "me siento feliz", "estoy feliz",
            "me siento contenta", "me siento contento",
            "estoy contenta", "estoy contento"
        ]
        if self.contiene_expresion(limpio, expresiones_positivas_directas):
            opciones_positivas = [
                "me alegra saber que te sentís bien. ¿Querés contarme qué te ayudó a sentirte así?",
                "qué bueno que te sientas bien. ¿Hubo algo lindo que quieras compartir?",
                "me alegra escuchar eso. ¿Qué fue lo mejor de tu día?"
            ]
            respuesta_positiva = random.choice(opciones_positivas)
            if nombre and nombre != "Invitad@":
                return f"{nombre}, {respuesta_positiva}"
            return respuesta_positiva[0].upper() + respuesta_positiva[1:]

        intenciones = DATOS_CONVERSACION["intenciones"]
        emociones = DATOS_CONVERSACION["emociones"]

        if self.contiene_expresion(limpio, intenciones["agradecimiento"]):
            return self.elegir_respuesta("agradecimiento", nombre)

        if self.contiene_expresion(limpio, intenciones["despedida"]):
            return self.elegir_respuesta("despedida", nombre)

        if self.contiene_expresion(limpio, intenciones["saludo"]):
            return self.elegir_respuesta("saludo", nombre)

        # Las negaciones tienen prioridad sobre la palabra positiva "bien".
        if self.detectar_negacion_emocional(limpio):
            return self.elegir_respuesta("negacion_emocional", nombre)

        if self.contiene_expresion(limpio, emociones["negativa"]):
            return self.elegir_respuesta("negativa", nombre)

        if self.contiene_expresion(limpio, emociones["positiva"]):
            return self.elegir_respuesta("positiva", nombre)

        if self.contiene_expresion(limpio, emociones["neutral"]):
            return self.elegir_respuesta("neutral", nombre)

        if "?" in original or limpio.startswith((
            "que ", "como ", "por que ", "cuando ", "donde ", "puedo "
        )):
            return self.elegir_respuesta("pregunta_general", nombre)

        return self.elegir_respuesta("general", nombre)



engine = EmotiChatEngine()




In [ ]:
# ==============================================================================
# 6. UTILIDADES Y ESTADO


In [ ]:
# ==============================================================================

def hora():
    return datetime.now(ZoneInfo("America/Argentina/Buenos_Aires")).strftime("%H:%M")


def saludo_del_dia():
    h = datetime.now(ZoneInfo("America/Argentina/Buenos_Aires")).hour
    if 6 <= h < 12:
        return "Buenos días"
    if 12 <= h < 20:
        return "Buenas tardes"
    return "Buenas noches"


def normalizar_nombre_texto(texto):
    texto = str(texto or "").strip().lower()
    texto = unicodedata.normalize("NFD", texto)
    texto = "".join(
        caracter for caracter in texto
        if unicodedata.category(caracter) != "Mn"
    )
    texto = re.sub(r"\s+", " ", texto)
    return texto.strip()


def limpiar_nombre(nombre):
    nombre = re.sub(
        r"[^a-zA-ZáéíóúÁÉÍÓÚñÑüÜ' -]",
        "",
        str(nombre or "")
    )
    nombre = re.sub(r"\s+", " ", nombre).strip(" -'")

    if not nombre:
        return ""

    return " ".join(
        palabra.capitalize()
        for palabra in nombre.split()
    )[:24]


def extraer_nombre(texto):
    """Extrae el nombre aunque venga dentro de una frase conversacional."""
    original = str(texto or "").strip()
    if not original:
        return ""

    normalizado = normalizar_nombre_texto(original)
    expresiones_invalidas = {
        "hola", "holaa", "holaaa", "buenas", "buen dia", "buenos dias",
        "buenas tardes", "buenas noches", "hey", "ey", "hi",
        "como estas", "todo bien", "si", "no", "ok", "okay", "dale",
        "gracias", "bien", "mal", "yo", "aca", "aqui", "emotichat"
    }

    if normalizado in expresiones_invalidas:
        return ""

    patrones = [
        r"(?:^|[,.!?¿¡]\s*)(?:hola\s+)?(?:yo\s+)?soy\s+([a-zA-ZáéíóúÁÉÍÓÚñÑüÜ' -]{2,40}?)(?=\s+(?:y|pero|aunque|estoy|me\s+siento|hoy|porque|que)\b|[,.!?¿¡]|$)",
        r"(?:^|[,.!?¿¡]\s*)(?:hola\s+)?me\s+llamo\s+([a-zA-ZáéíóúÁÉÍÓÚñÑüÜ' -]{2,40}?)(?=\s+(?:y|pero|aunque|estoy|me\s+siento|hoy|porque|que)\b|[,.!?¿¡]|$)",
        r"(?:^|[,.!?¿¡]\s*)(?:hola\s+)?mi\s+nombre\s+es\s+([a-zA-ZáéíóúÁÉÍÓÚñÑüÜ' -]{2,40}?)(?=\s+(?:y|pero|aunque|estoy|me\s+siento|hoy|porque|que)\b|[,.!?¿¡]|$)",
        r"(?:^|[,.!?¿¡]\s*)(?:podes|podés|puedes)\s+llamarme\s+([a-zA-ZáéíóúÁÉÍÓÚñÑüÜ' -]{2,40}?)(?=\s+(?:y|pero|aunque|estoy|me\s+siento|hoy|porque|que)\b|[,.!?¿¡]|$)",
        r"(?:^|[,.!?¿¡]\s*)llamame\s+([a-zA-ZáéíóúÁÉÍÓÚñÑüÜ' -]{2,40}?)(?=\s+(?:y|pero|aunque|estoy|me\s+siento|hoy|porque|que)\b|[,.!?¿¡]|$)",
    ]

    candidato = ""
    for patron in patrones:
        coincidencia = re.search(patron, original, flags=re.IGNORECASE)
        if coincidencia:
            candidato = limpiar_nombre(coincidencia.group(1))
            break

    if not candidato:
        partes = original.strip(" ,.!?¿¡").split()
        if len(partes) not in (1, 2):
            return ""
        candidato = limpiar_nombre(" ".join(partes))

    candidato_normalizado = normalizar_nombre_texto(candidato)
    if candidato_normalizado in expresiones_invalidas:
        return ""
    if not 2 <= len(candidato) <= 24:
        return ""
    if len(candidato.split()) > 2:
        return ""

    return candidato


def extraer_mensaje_despues_del_nombre(texto, nombre):
    """Recupera lo que la persona dijo después de presentarse."""
    original = str(texto or "").strip()
    if not original or not nombre:
        return ""

    patrones = [
        rf"^\s*(?:hola[, ]*)?(?:yo\s+)?soy\s+{re.escape(nombre)}\s*(?:,|\by\b)?\s*(.*)$",
        rf"^\s*(?:hola[, ]*)?me\s+llamo\s+{re.escape(nombre)}\s*(?:,|\by\b)?\s*(.*)$",
        rf"^\s*(?:hola[, ]*)?mi\s+nombre\s+es\s+{re.escape(nombre)}\s*(?:,|\by\b)?\s*(.*)$",
    ]

    for patron in patrones:
        coincidencia = re.match(patron, original, flags=re.IGNORECASE)
        if coincidencia:
            return coincidencia.group(1).strip(" ,.!?¿¡")

    return ""

def parece_mensaje_conversacional(texto):
    """Distingue una frase emocional completa de una presentación personal."""
    normalizado = normalizar_nombre_texto(texto)
    palabras = normalizado.split()
    marcadores = (
        "me siento", "estoy ", "hoy ", "me pasa", "tengo ",
        "quiero ", "necesito ", "me preocupa", "me alegra",
        "me duele", "no estoy", "no me siento"
    )
    saludo_con_contenido = normalizado.startswith((
        "hola ", "buen dia ", "buenos dias ",
        "buenas tardes ", "buenas noches "
    ))
    return len(palabras) >= 3 and (
        saludo_con_contenido
        or any(marcador in normalizado for marcador in marcadores)
    )


def validar_nombre(texto):
    nombre = extraer_nombre(texto)

    if not nombre:
        return (
            False,
            "",
            "No pude identificar un nombre. Podés escribirlo, por ejemplo: María, o contarme directamente cómo te sentís."
        )

    return True, nombre, ""


def estado_inicial():
    return [{
        "rol": "bot",
        "texto": (
            "Hola. Podés decirme tu nombre o contarme directamente cómo te sentís."
        ),
        "hora": hora()
    }]


def inicial_usuario(nombre):
    if not nombre or nombre == "Invitad@":
        return "T"
    return html.escape(nombre[0].upper())


def logo_html(clase="logo-img"):
    if LOGO_URL:
        return f'<img class="{clase}" src="{LOGO_URL}" alt="EmotiChat logo">'
    return '<div class="logo-fallback">EC</div>'




In [ ]:
# ==============================================================================
# 7. COMPONENTES DE PRESENTACIÓN


In [ ]:
# ==============================================================================

def render_topbar(nombre):
    titulo = f"Hola, {html.escape(nombre)}" if nombre and nombre != "Invitad@" else saludo_del_dia()

    return f"""
    <header class="topbar">
        <div class="top-left">
            <div class="top-logo">{logo_html("top-logo-img")}</div>
            <div>
                <h2>{titulo}</h2>
                <p>Tu espacio de bienestar</p>
            </div>
        </div>

    </header>
    """


def render_chat(historial, nombre="Invitad@", typing=False):
    mensajes = '<div class="day-pill">Hoy</div>'

    for item in historial:
        if item.get("html", False):
            texto = item["texto"]
        else:
            texto = html.escape(item["texto"]).replace("\n", "<br>")
        h = item.get("hora", hora())

        if item["rol"] == "bot":
            mensajes += f"""
            <div class="msg bot-msg">
                <div class="avatar bot-avatar">{logo_html("avatar-logo")}</div>
                <div class="bubble bot-bubble">
                    <p>{texto}</p>
                    <small>{h}</small>
                </div>
            </div>
            """
        else:
            mensajes += f"""
            <div class="msg user-msg">
                <div class="bubble user-bubble">
                    <p>{texto}</p>
                    <small>{h} ✓✓</small>
                </div>
                <div class="avatar user-avatar">{inicial_usuario(nombre)}</div>
            </div>
            """

    if typing:
        mensajes += f"""
        <div class="msg bot-msg">
            <div class="avatar bot-avatar">{logo_html("avatar-logo")}</div>
            <div class="bubble bot-bubble typing-bubble">
                <p>EmotiChat está escribiendo</p>
                <div class="typing-dots"><span></span><span></span><span></span></div>
            </div>
        </div>
        """

    mensajes += '<div id="chat-end" aria-hidden="true"></div>'
    return f'<div class="messages" role="log" aria-live="polite">{mensajes}</div>'


ICONOS_SVG = {
    "microphone": """<svg viewBox="0 0 24 24" aria-hidden="true"><rect x="9" y="3" width="6" height="11" rx="3"></rect><path d="M5 11a7 7 0 0 0 14 0M12 18v3M8 21h8"></path></svg>""",
    "speaker": """<svg viewBox="0 0 24 24" aria-hidden="true"><path d="M4 9h4l5-4v14l-5-4H4z"></path><path d="M16 9a4 4 0 0 1 0 6M18.5 6.5a8 8 0 0 1 0 11"></path></svg>""",
    "waveform": """<svg viewBox="0 0 24 24" aria-hidden="true"><path d="M4 10v4M8 7v10M12 4v16M16 7v10M20 10v4"></path></svg>""",
    "heart": """<svg viewBox="0 0 24 24" aria-hidden="true"><path d="M20.8 4.6a5.5 5.5 0 0 0-7.8 0L12 5.7l-1.1-1.1a5.5 5.5 0 0 0-7.8 7.8L12 21l8.8-8.6a5.5 5.5 0 0 0 0-7.8z"></path></svg>""",
    "globe": """<svg viewBox="0 0 24 24" aria-hidden="true"><circle cx="12" cy="12" r="9"></circle><path d="M3 12h18M12 3c2.4 2.5 3.6 5.5 3.6 9S14.4 18.5 12 21M12 3C9.6 5.5 8.4 8.5 8.4 12S9.6 18.5 12 21"></path></svg>""",
    "confidence": """<svg viewBox="0 0 24 24" aria-hidden="true"><path d="M12 3l7 3v5c0 4.6-2.8 8.1-7 10-4.2-1.9-7-5.4-7-10V6z"></path><path d="m9 12 2 2 4-5"></path></svg>""",
}


def icono_svg(nombre):
    """Devuelve un ícono vectorial estable para las tarjetas de estado."""
    return ICONOS_SVG.get(nombre, ICONOS_SVG["waveform"])


def card_html(icono, titulo, valor, detalle, clase, progreso=None):
    barra = ""
    if progreso is not None:
        barra = f'<div class="progress"><div style="width:{progreso}%"></div></div>'

    return f"""
    <div class="summary-card {clase}">
        <div class="summary-icon">{icono}</div>
        <div class="summary-copy">
            <span>{titulo}</span>
            <h3>{valor}</h3>
            <p>{detalle}</p>
            {barra}
        </div>
    </div>
    """


def panel_inicial():
    return """
    <div class="summary-panel">
        <div class="panel-title">
            <span class="section-mark" aria-hidden="true"></span>
            <div>
                <h2>Conversación</h2>
                <p>Estado de la sesión actual</p>
            </div>
        </div>
        """ + card_html(icono_svg("heart"), "Estado emocional", "Esperando inicio", "Primero registramos tu nombre.", "green-card") + """
        """ + card_html(icono_svg("globe"), "Idioma detectado", "—", "Se mostrará al enviar un mensaje.", "purple-card") + """
        """ + card_html(icono_svg("confidence"), "Confianza del análisis", "—", "Estimación del procesamiento.", "blue-card", 0) + """
        """ + card_html(icono_svg("waveform"), "Calidad del audio", "Pendiente", "Disponible al procesar audio.", "orange-card") + """
    </div>
    """


def panel_sesion(nombre):
    return """
    <div class="summary-panel">
        <div class="panel-title">
            <span class="section-mark" aria-hidden="true"></span>
            <div>
                <h2>Conversación</h2>
                <p>Estado de la sesión actual</p>
            </div>
        </div>
        """ + card_html(icono_svg("heart"), "Estado emocional", "Sesión activa", f"Hola, {html.escape(nombre)}.", "green-card") + """
        """ + card_html(icono_svg("globe"), "Idioma detectado", "—", "Esperando mensaje.", "purple-card") + """
        """ + card_html(icono_svg("confidence"), "Confianza del análisis", "—", "Estimación del procesamiento.", "blue-card", 0) + """
        """ + card_html(icono_svg("waveform"), "Calidad del audio", "Pendiente", "Disponible al procesar audio.", "orange-card") + """
    </div>
    """


def obtener_contexto_usuario(historial, limite=3):
    """Une los últimos mensajes del usuario para conservar el contexto emocional."""
    mensajes = [
        item.get("texto", "").strip()
        for item in (historial or [])
        if item.get("rol") == "user" and item.get("texto", "").strip()
    ]
    return ". ".join(mensajes[-limite:])


def calcular_consenso_sentimientos(tb, vd, tr):
    resultados = [
        tb.get("sentimiento"),
        vd.get("sentimiento"),
        tr.get("sentimiento")
    ]
    validos = [
        resultado for resultado in resultados
        if resultado in {"Positivo", "Negativo", "Neutro"}
    ]

    if not validos:
        return "No disponible", 0

    conteos = {
        sentimiento: validos.count(sentimiento)
        for sentimiento in {"Positivo", "Negativo", "Neutro"}
    }
    consenso = max(conteos, key=conteos.get)
    coincidencias = conteos[consenso]

    confianza_modelos = round((coincidencias / len(validos)) * 100)
    confianza_transformer = round(float(tr.get("confianza", 0)) * 100)

    if confianza_transformer:
        confianza = round((confianza_modelos * 0.65) + (confianza_transformer * 0.35))
    else:
        confianza = confianza_modelos

    return consenso, max(0, min(100, confianza))


def panel_analisis(historial, tiempo_total, tipo_entrada="Texto"):
    texto = obtener_contexto_usuario(historial)

    if not texto:
        return panel_inicial()

    traduccion = engine.traduccion_encadenada(texto)
    tb = engine.sentimiento_textblob(texto)
    vd = engine.sentimiento_vader(texto)
    tr = engine.sentimiento_transformers(texto)

    consenso, confianza = calcular_consenso_sentimientos(tb, vd, tr)

    idioma_codigo = traduccion.get("idioma_origen", "—")
    idioma = NOMBRES_IDIOMAS.get(idioma_codigo, str(idioma_codigo).upper())

    if consenso == "Positivo":
        estado = "Positivo"
        icono = "☺"
        clase_estado = "positive-status"
    elif consenso == "Negativo":
        estado = "Necesita atención"
        icono = "●"
        clase_estado = "negative-status"
    elif consenso == "Neutro":
        estado = "Neutro"
        icono = "○"
        clase_estado = "neutral-status"
    else:
        estado = "No disponible"
        icono = "—"
        clase_estado = "neutral-status"

    entrada_detalle = (
        f"Tiempo total: {round(tiempo_total, 2)} s"
        if tipo_entrada == "Texto"
        else f"Audio procesado en {round(tiempo_total, 2)} s"
    )

    return f"""
    <div class="summary-panel compact-analysis">
        <div class="panel-title">
            <span class="section-mark" aria-hidden="true"></span>
            <div>
                <h2>Análisis de la conversación</h2>
                <p>Se consideran los últimos mensajes para conservar el contexto.</p>
            </div>
        </div>

        <div class="analysis-overview {clase_estado}">
            <span class="overview-icon">{icono}</span>
            <div>
                <span>Estado emocional</span>
                <h3>{estado}</h3>
                <p>Resultado por consenso entre TextBlob, VADER y Transformers.</p>
            </div>
        </div>

        <div class="analysis-inline-metrics">
            <div><span>Idioma</span><strong>{html.escape(str(idioma))}</strong></div>
            <div><span>Confianza</span><strong>{confianza}%</strong></div>
            <div><span>Entrada</span><strong>{html.escape(tipo_entrada)}</strong></div>
        </div>

        <details class="analysis-details">
            <summary>Ver análisis técnico completo</summary>
            <p><b>Contexto analizado:</b> {html.escape(texto)}</p>
            <p><b>TextBlob:</b> {tb.get("sentimiento", "—")} | Polaridad: {tb.get("polaridad", 0)} | Subjetividad: {tb.get("subjetividad", 0)}</p>
            <p><b>VADER:</b> {vd.get("sentimiento", "—")} | Compound: {vd.get("compound", 0)}</p>
            <p><b>Transformers:</b> {tr.get("sentimiento", "—")} | Confianza: {round(float(tr.get("confianza", 0)) * 100, 1)}%</p>
            <p><b>Retrotraducción:</b> {html.escape(str(traduccion.get("final", "—")))}</p>
            <p><b>Procesamiento:</b> {entrada_detalle}</p>
        </details>
    </div>
    """


def sidebar_info(titulo, texto):
    return f"""
    <div class="sidebar-info-card">
        <h3>{titulo}</h3>
        <p>{texto}</p>
    </div>
    """


def sidebar_conversacion(nombre, etapa):
    if etapa == "nombre":
        return sidebar_info(
            "Conversación",
            "El chat está esperando que ingreses el nombre para iniciar la sesión."
        )
    return sidebar_info(
        "Conversación",
        f"Sesión activa para {html.escape(nombre)}. Puedes continuar escribiendo en el chat central."
    )


def sidebar_audio():
    return sidebar_info(
        "Procesamiento de voz",
        "Desde el botón Audio puedes trabajar con la respuesta en audio generada por la aplicación."
    )


def sidebar_traduccion():
    return sidebar_info(
        "Traducción",
        "El sistema detecta el idioma del mensaje, traduce y realiza retrotraducción para comparar variaciones del texto."
    )


def sidebar_sentimientos():
    return sidebar_info(
        "Sentimientos",
        "El análisis combina TextBlob y VADER para estimar polaridad, subjetividad y estado emocional del mensaje."
    )


def panel_modulo(titulo, subtitulo, tarjetas):
    return f"""
    <div id="panel-modulo" class="summary-panel module-summary-panel">
        <div class="panel-title">
            <span class="section-mark" aria-hidden="true"></span>
            <div>
                <h2>{titulo}</h2>
                <p>{subtitulo}</p>
            </div>
        </div>
        {tarjetas}
    </div>
    """


def ultimo_mensaje_usuario(historial):
    if not historial:
        return ""
    for item in reversed(historial):
        if item.get("rol") == "user":
            return item.get("texto", "")
    return ""


def info_box(titulo, texto):
    return f"""
    <div class="info-box">
        <h4>{titulo}</h4>
        <p>{texto}</p>
    </div>
    """


def panel_audio(historial=None):
    """La vista de audio muestra directamente sus herramientas operativas."""
    return ""



def normalizar_retrotraduccion(texto):
    """
    Normaliza palabras y expresiones equivalentes frecuentes antes de comparar.
    Esto evita penalizar cambios de redacción como "ando" por "estoy".
    """
    palabras = normalizar_para_similitud(texto).split()

    equivalencias = {
        "ando": "estoy",
        "encuentro": "estoy",
        "hallaba": "estaba",
        "hallado": "estado",
        "tengo": "siento",
        "poseo": "tengo",
        "contenta": "feliz",
        "contento": "feliz",
        "alegre": "feliz",
        "apenada": "triste",
        "apenado": "triste",
        "fatigada": "cansada",
        "fatigado": "cansado",
        "enferma": "resfriada",
        "enfermo": "resfriado"
    }

    return [equivalencias.get(palabra, palabra) for palabra in palabras]


def calcular_similitud_retrotraduccion(original, final):
    """
    Compara secuencias de palabras normalizadas.

    Se utiliza una comparación léxica interpretable, adecuada para mostrar
    cambios de redacción sin confundirlos automáticamente con pérdida de sentido.
    """
    original_palabras = normalizar_retrotraduccion(original)
    final_palabras = normalizar_retrotraduccion(final)

    if not original_palabras or not final_palabras:
        return 0

    similitud_secuencia = difflib.SequenceMatcher(
        None,
        original_palabras,
        final_palabras
    ).ratio()

    conjunto_original = set(original_palabras)
    conjunto_final = set(final_palabras)
    union = conjunto_original | conjunto_final
    similitud_vocabulario = (
        len(conjunto_original & conjunto_final) / len(union)
        if union else 0
    )

    # La secuencia tiene mayor peso; el vocabulario actúa como apoyo.
    porcentaje = (0.75 * similitud_secuencia + 0.25 * similitud_vocabulario) * 100
    return round(porcentaje)


def analizar_retrotraduccion(original, final):
    original_norm = normalizar_para_similitud(original)
    final_norm = normalizar_para_similitud(final)
    similitud = calcular_similitud_retrotraduccion(original, final)

    original_palabras = set(normalizar_retrotraduccion(original))
    final_palabras = set(normalizar_retrotraduccion(final))
    cambios = sorted(original_palabras ^ final_palabras)[:8]
    detalle = ", ".join(cambios) if cambios else "Cambios mínimos de redacción"

    if similitud >= 90:
        estado, nivel = "Excelente", "Alta similitud"
        explicacion = "La retraducción conserva prácticamente el mismo significado del mensaje. Las diferencias corresponden a sinónimos o cambios naturales de redacción."
    elif similitud >= 75:
        estado, nivel = "Buena", "Similitud intermedia"
        explicacion = "Se observaron pequeños cambios de redacción sin alterar la idea principal del mensaje."
    else:
        estado, nivel = "Requiere revisión", "Similitud baja"
        explicacion = "La retraducción presenta diferencias importantes. Conviene revisar ambas versiones para confirmar que el significado se haya mantenido."

    return {
        "similitud": similitud,
        "estado": estado,
        "nivel": nivel,
        "explicacion": explicacion,
        "cambios": detalle
    }

def panel_traduccion_menu(historial):
    texto = ultimo_mensaje_usuario(historial)
    if not texto:
        tarjetas = card_html(icono_svg("globe"), "Traducción", "Esperando mensaje", "Escribí algo en el chat y luego abrí este módulo.", "purple-card") + info_box("Función del módulo", "Detecta el idioma, traduce el mensaje y realiza una retrotraducción para comprobar si el significado se conserva.")
        return panel_modulo("Traducción", "Traducción automática y comparación entre idiomas", tarjetas)

    traduccion = engine.traduccion_encadenada(texto)
    if not traduccion.get("disponible", False):
        tarjetas = (
            f'<div class="module-card wide-card compact-content"><h3>Texto original</h3><p>{html.escape(texto)}</p></div>'
            + info_box(
                "Traducción temporalmente no disponible",
                "Los servicios externos no respondieron. Esperá unos segundos y volvé a abrir este módulo. El mensaje original se conserva sin modificaciones."
            )
        )
        return panel_modulo(
            "Traducción",
            "No fue posible conectar con los servicios de traducción",
            tarjetas
        )
    idioma = NOMBRES_IDIOMAS.get(traduccion["idioma_origen"], traduccion["idioma_origen"])
    confianza_idioma = traduccion.get("confianza_idioma", "Confiable")
    analisis = analizar_retrotraduccion(traduccion["original"], traduccion["final"])
    pasos_html = "".join(
        f"<div class='translation-step'><span>{html.escape(paso['idioma'])}</span><p>{html.escape(paso['texto'])}</p></div>"
        for paso in traduccion["pasos"]
    )
    tarjetas = (
        f'<div class="result-grid translation-overview">'
        f'<div class="result-card"><span>Idioma detectado</span><strong>{html.escape(idioma)}</strong><small>{html.escape(confianza_idioma)}</small></div>'
        f'<div class="result-card"><span>Similitud</span><strong>{analisis["similitud"]} %</strong></div>'
        f'<div class="result-card"><span>Estado</span><strong>{analisis["estado"]}</strong></div>'
        f'<div class="result-card"><span>Tecnología</span><strong>Google + MyMemory</strong></div></div>'
        f'<div class="module-card wide-card compact-content"><h3>Texto original</h3><p>{html.escape(traduccion["original"])}</p></div>'
        f'<div class="module-card wide-card compact-content"><h3>Recorrido de traducción</h3>{pasos_html}</div>'
        f'<div class="module-card wide-card compact-content"><h3>Retrotraducción</h3><p>{html.escape(traduccion["final"])}</p></div>'
        f'<div class="interpretation-card"><div class="interpretation-header"><span>Análisis de la retrotraducción</span><b>{analisis["nivel"]}</b></div>'
        f'<div class="progress-track"><span style="width:{analisis["similitud"]}%"></span></div>'
        f'<div class="interpretation-grid"><div><h4>¿Qué ocurrió?</h4><p>{analisis["explicacion"]}</p></div>'
        f'<div><h4>Cambios observados</h4><p>{html.escape(analisis["cambios"])}</p></div></div>'
        '<p class="technology-note"><strong>Tecnologías:</strong> GoogleTranslator realiza la traducción y MyMemory actúa como respaldo; langdetect identifica automáticamente el idioma original.</p></div>'
    )
    return panel_modulo("Traducción", "Traducción automática y comparación entre idiomas", tarjetas)


def clasificar_valor_sentimiento(valor):
    """
    Convierte un valor entre -1 y 1 en una categoría fácil de interpretar.
    Se usa para explicar tanto la polaridad de TextBlob como el compound de VADER.
    """
    if valor >= 0.60:
        return "Muy positivo", "El mensaje transmite una emoción claramente positiva."
    if valor >= 0.20:
        return "Levemente positivo", "El mensaje presenta una tendencia positiva moderada."
    if valor > -0.20:
        return "Neutro", "No se identifica una emoción positiva o negativa predominante."
    if valor > -0.60:
        return "Levemente negativo", "El mensaje presenta una tendencia negativa moderada."
    return "Muy negativo", "El mensaje transmite una emoción claramente negativa."


def interpretar_subjetividad(valor):
    """
    Explica el valor de subjetividad de TextBlob en lenguaje sencillo.
    """
    if valor < 0.35:
        return "Predominan expresiones objetivas o descriptivas."
    if valor < 0.70:
        return "Hay una mezcla de información y opiniones o emociones personales."
    return "Predominan opiniones, sensaciones o emociones personales."


def construir_tabla_comparativa_sentimientos(tb, vd, tr):
    categoria_tb, explicacion_tb = clasificar_valor_sentimiento(tb["polaridad"])
    categoria_vd, explicacion_vd = clasificar_valor_sentimiento(vd["compound"])

    filas = [
        (
            "TextBlob",
            tb["sentimiento"],
            f'{tb["polaridad"]:.3f}',
            categoria_tb,
            explicacion_tb
        ),
        (
            "VADER",
            vd["sentimiento"],
            f'{vd["compound"]:.3f}',
            categoria_vd,
            explicacion_vd
        ),
        (
            "Transformers",
            tr["sentimiento"],
            f'{tr["confianza"] * 100:.1f} %',
            "Confianza del modelo",
            "Modelo neuronal basado en Transformers."
        ),
    ]

    cuerpo = "".join(
        f"""
        <tr>
            <td><strong>{html.escape(metodo)}</strong></td>
            <td>{html.escape(resultado)}</td>
            <td>{html.escape(valor)}</td>
            <td>{html.escape(categoria)}</td>
            <td>{html.escape(explicacion)}</td>
        </tr>
        """
        for metodo, resultado, valor, categoria, explicacion in filas
    )

    return f"""
    <table class="metric-table sentiment-comparison-table">
        <thead>
            <tr>
                <th>Método</th>
                <th>Resultado</th>
                <th>Valor o confianza</th>
                <th>Referencia</th>
                <th>Explicación</th>
            </tr>
        </thead>
        <tbody>{cuerpo}</tbody>
    </table>
    """



def describir_acuerdo_modelos(tb, vd, tr):
    """Resume el nivel de acuerdo entre TextBlob, VADER y Transformers."""
    resultados = {
        "TextBlob": tb.get("sentimiento", "No disponible"),
        "VADER": vd.get("sentimiento", "No disponible"),
        "Transformers": tr.get("sentimiento", "No disponible")
    }
    validos = {
        metodo: resultado
        for metodo, resultado in resultados.items()
        if resultado in {"Positivo", "Negativo", "Neutro"}
    }

    if not validos:
        return "No disponible", "No fue posible obtener resultados válidos de los modelos.", 0

    conteos = {
        sentimiento: list(validos.values()).count(sentimiento)
        for sentimiento in {"Positivo", "Negativo", "Neutro"}
    }
    consenso = max(conteos, key=conteos.get)
    coincidencias = conteos[consenso]
    total = len(validos)

    if coincidencias == total and total == 3:
        nivel = "Alta"
        explicacion = (
            f"Los tres modelos coinciden en una clasificación {consenso.lower()}."
        )
    elif coincidencias >= 2:
        nivel = "Media"
        discrepantes = [
            metodo for metodo, resultado in validos.items()
            if resultado != consenso
        ]
        detalle = ", ".join(discrepantes) if discrepantes else "un modelo"
        explicacion = (
            f"Dos de los tres modelos coinciden en una clasificación {consenso.lower()}; "
            f"{detalle} presenta una interpretación diferente."
        )
    else:
        nivel = "Baja"
        explicacion = (
            "Los modelos no alcanzan una mayoría clara. El texto puede ser breve, "
            "ambiguo o contener emociones mixtas."
        )

    _, porcentaje = calcular_consenso_sentimientos(tb, vd, tr)
    return nivel, explicacion, porcentaje


def generar_conclusion_tres_modelos(tb, vd, tr, consenso, nivel_confianza):
    """Genera una conclusión usando el consenso de los tres modelos."""
    subjetividad = float(tb.get("subjetividad", 0) or 0)

    if consenso == "Positivo":
        inicio = "El contexto reciente transmite una tendencia emocional predominantemente positiva."
    elif consenso == "Negativo":
        inicio = "El contexto reciente presenta una tendencia emocional predominantemente negativa."
    elif consenso == "Neutro":
        inicio = "En el contexto reciente no se detecta una emoción claramente predominante."
    else:
        inicio = "No fue posible establecer una tendencia emocional predominante."

    acuerdo = {
        "Alta": " Los tres modelos coinciden, por lo que el acuerdo entre métodos es alto.",
        "Media": " Existe una mayoría entre los modelos, aunque uno presenta una lectura diferente.",
        "Baja": " Los modelos muestran resultados diferentes, por lo que la interpretación debe tomarse con cautela.",
        "No disponible": " No hay suficientes resultados válidos para comparar los métodos."
    }.get(nivel_confianza, "")

    if subjetividad < 0.35:
        cierre = " Predominan expresiones descriptivas u objetivas."
    elif subjetividad < 0.70:
        cierre = " El texto combina información con emociones u opiniones personales."
    else:
        cierre = " Predominan emociones, sensaciones u opiniones personales."

    return inicio + acuerdo + cierre


def generar_recomendacion_consenso(consenso):
    """Devuelve una recomendación orientativa coherente con el consenso."""
    if consenso == "Negativo":
        return (
            "El análisis detecta una tendencia negativa en los últimos mensajes. "
            "Puede ser útil expresar lo que sentís, hablar con alguien de confianza "
            "o buscar apoyo profesional si el malestar se mantiene o aumenta."
        )
    if consenso == "Positivo":
        return (
            "El contexto refleja una tendencia positiva. Podés identificar qué situaciones "
            "o vínculos están contribuyendo a este bienestar para intentar sostenerlos."
        )
    if consenso == "Neutro":
        return (
            "No se detecta una emoción predominante. Podés continuar la conversación "
            "para aportar más contexto y obtener una interpretación más completa."
        )
    return (
        "No hay suficiente información para generar una recomendación específica. "
        "Conviene continuar la conversación y considerar el contexto completo."
    )


def panel_sentimientos_menu(historial):
    texto = obtener_contexto_usuario(historial, limite=3)

    if not texto:
        tarjetas = (
            card_html(
                "🧠",
                "Análisis de sentimientos",
                "Esperando mensajes",
                "Escribí en el chat y luego abrí este módulo.",
                "green-card"
            ) +
            info_box(
                "¿Qué hace este módulo?",
                "Analiza los últimos tres mensajes del usuario con TextBlob, VADER y Transformers, "
                "y compara sus resultados para obtener un consenso orientativo."
            )
        )
        return panel_modulo(
            "Análisis de sentimientos",
            "Comparación entre TextBlob, VADER y Transformers",
            tarjetas
        )

    tb = engine.sentimiento_textblob(texto)
    vd = engine.sentimiento_vader(texto)
    tr = engine.sentimiento_transformers(texto)

    consenso, porcentaje_confianza = calcular_consenso_sentimientos(tb, vd, tr)
    confianza, explicacion_confianza, _ = describir_acuerdo_modelos(tb, vd, tr)
    conclusion = generar_conclusion_tres_modelos(tb, vd, tr, consenso, confianza)
    recomendacion = generar_recomendacion_consenso(consenso)

    if consenso == "Positivo":
        icono_resultado = "☺"
        clase_tarjeta = "green-card"
    elif consenso == "Negativo":
        icono_resultado = "☁"
        clase_tarjeta = "orange-card"
    elif consenso == "Neutro":
        icono_resultado = "○"
        clase_tarjeta = "blue-card"
    else:
        icono_resultado = "⚠"
        clase_tarjeta = "purple-card"

    tabla_comparativa = construir_tabla_comparativa_sentimientos(tb, vd, tr)
    subjetividad_texto = interpretar_subjetividad(tb.get("subjetividad", 0))

    tabla_referencias = """
    <table class="metric-table">
        <thead>
            <tr>
                <th>Rango del valor</th>
                <th>Interpretación sencilla</th>
            </tr>
        </thead>
        <tbody>
            <tr><td>0.60 a 1.00</td><td>Emoción claramente positiva.</td></tr>
            <tr><td>0.20 a 0.59</td><td>Emoción levemente positiva.</td></tr>
            <tr><td>-0.19 a 0.19</td><td>Estado neutral o sin emoción predominante.</td></tr>
            <tr><td>-0.59 a -0.20</td><td>Emoción levemente negativa.</td></tr>
            <tr><td>-1.00 a -0.60</td><td>Emoción claramente negativa.</td></tr>
        </tbody>
    </table>
    """

    tarjetas = f"""
        {card_html(
            icono_resultado,
            "Consenso de los tres modelos",
            html.escape(consenso),
            html.escape(explicacion_confianza),
            clase_tarjeta
        )}

        {card_html(
            "✓",
            "Confianza del consenso",
            html.escape(confianza),
            f"Acuerdo combinado estimado: {porcentaje_confianza}%.",
            "blue-card",
            porcentaje_confianza
        )}

        <div class="module-card wide-card">
            <h3>Contexto analizado</h3>
            <p>
                Se analizaron hasta los últimos tres mensajes del usuario para evitar
                que el resultado dependa únicamente de una frase aislada.
            </p>
            <div class="context-preview">{html.escape(texto)}</div>
        </div>

        <div class="module-card wide-card">
            <h3>Comparación de métodos</h3>
            <p>
                El mismo contexto se procesa con TextBlob, VADER y Transformers.
                El consenso se obtiene por mayoría y considera la confianza del modelo neuronal.
            </p>
            {tabla_comparativa}
        </div>

        <div class="module-card wide-card">
            <h3>Detalle de TextBlob</h3>
            <p>
                <strong>Polaridad:</strong> {tb.get("polaridad", 0)} —
                indica si el tono tiende a negativo, neutral o positivo.
            </p>
            <p>
                <strong>Subjetividad:</strong> {tb.get("subjetividad", 0)} —
                {html.escape(subjetividad_texto)}
            </p>
        </div>

        <div class="module-card wide-card">
            <h3>Valores de referencia</h3>
            <p>
                Las escalas de TextBlob y VADER se expresan entre -1 y 1.
                Transformers informa una etiqueta y la confianza propia del modelo.
            </p>
            {tabla_referencias}
        </div>

        <div class="module-card wide-card">
            <h3>¿Por qué se usan tres métodos?</h3>
            <p><strong>TextBlob</strong> aporta polaridad y subjetividad.</p>
            <p><strong>VADER</strong> está orientado a mensajes breves y lenguaje conversacional.</p>
            <p><strong>Transformers</strong> incorpora un modelo neuronal capaz de captar patrones más complejos.</p>
            <p>
                Compararlos permite detectar acuerdos y discrepancias sin depender de una sola herramienta.
            </p>
        </div>

        <div class="module-card wide-card">
            <h3>Conclusión del análisis</h3>
            <p>{html.escape(conclusion)}</p>
        </div>

        <div class="module-card wide-card">
            <h3>Recomendación orientativa</h3>
            <p>{html.escape(recomendacion)}</p>
        </div>

        {info_box(
            "Aclaración importante",
            "Este análisis es una estimación automática del tono del texto. "
            "No constituye un diagnóstico psicológico, psiquiátrico o clínico."
        )}

        <div class="tech-pill">Procesado con: TextBlob • VADER • Transformers</div>
    """

    return panel_modulo(
        "Análisis de sentimientos",
        "Consenso de tres modelos sobre los últimos tres mensajes",
        tarjetas
    )


def tabla_comparacion_bilingue(comparacion):
    if comparacion.get("error"):
        return info_box("Comparación no disponible", comparacion["error"])
    filas = []
    for metodo, datos in comparacion.get("resultados", {}).items():
        es, en = datos["es"], datos["en"]
        metrica = datos["metrica"]
        val_es = es.get(metrica, 0)
        val_en = en.get(metrica, 0)
        if metrica == "confianza":
            val_es = f"{float(val_es)*100:.1f} %"
            val_en = f"{float(val_en)*100:.1f} %"
        filas.append(f"""
        <tr>
            <td><strong>{html.escape(metodo)}</strong></td>
            <td>{html.escape(es.get('sentimiento','—'))}<br><small>{metrica}: {val_es}</small></td>
            <td>{html.escape(en.get('sentimiento','—'))}<br><small>{metrica}: {val_en}</small></td>
            <td>{html.escape(datos.get('analisis',''))}</td>
        </tr>""")
    return f"""
    <div class="module-card wide-card">
      <h3>🌐 Comparación de sentimientos: español e inglés</h3>
      <p><strong>Traducción inglesa:</strong> {html.escape(comparacion.get('texto_ingles',''))}</p>
      <div style="overflow-x:auto">
      <table class="metric-table">
        <thead><tr><th>Método</th><th>Español</th><th>Inglés</th><th>Diferencia</th></tr></thead>
        <tbody>{''.join(filas)}</tbody>
      </table></div>
    </div>"""


def investigacion_critica_html(comparacion):
    cambios = sum(1 for d in comparacion.get("resultados", {}).values() if not d.get("coinciden"))
    return f"""
    <div class="module-card wide-card">
      <h3>🔎 Investigación crítica</h3>
      <ol>
        <li><strong>¿Por qué cambian los resultados?</strong> Los analizadores VADER, TextBlob y el modelo Transformers fueron entrenados principalmente con inglés. En español pueden no reconocer negaciones, intensificadores, modismos o matices; la traducción vuelve el texto más cercano a sus léxicos, pero también puede cambiar el tono. En este experimento, {cambios} de 3 métodos cambiaron de categoría entre idiomas.</li>
        <li><strong>Método más confiable para español:</strong> En este prototipo, Transformers ofrece contexto más amplio que los métodos léxicos, aunque el modelo utilizado está orientado al inglés. Para producción convendría un modelo entrenado específicamente con corpus en español.</li>
        <li><strong>Factores culturales y lingüísticos:</strong> modismos argentinos, diminutivos, intensidad, negación, ambigüedad, acento, palabras como “bárbaro”, “quilombo” o “re” y diferencias culturales en la expresión emocional.</li>
        <li><strong>Sarcasmo e ironía:</strong> los tres sistemas pueden fallar porque el significado literal contradice la intención. Se necesitarían contexto conversacional, señales de voz y modelos entrenados con ejemplos de sarcasmo.</li>
      </ol>
    </div>"""


def ejecutar_pruebas_audios_proyecto():
    try:
        filas = engine.comparar_audios_obligatorios()
        cuerpo = "".join(
            f"<tr><td>{html.escape(f['audio'])}</td><td>{html.escape(f['metodo'])}</td>"
            f"<td>{html.escape(f['transcripcion'])}</td><td>{f['tiempo']:.2f} s</td>"
            f"<td>{f['wer']:.1f} %</td><td>{f['precision']:.1f} %</td>"
            f"<td>{html.escape(f['observacion'])}</td></tr>"
            for f in filas
        )
        return f"""
        <div class="module-card wide-card">
          <h3>Resultados de los dos audios WAV de prueba</h3>
          <div style="overflow-x:auto"><table class="metric-table">
          <thead><tr><th>Audio</th><th>Método</th><th>Transcripción</th><th>Tiempo</th><th>WER</th><th>Precisión</th><th>Análisis</th></tr></thead>
          <tbody>{cuerpo}</tbody></table></div>
          <p><strong>Lectura del WER:</strong> 0 % indica una transcripción perfecta. Cuanto menor sea el porcentaje, menos errores de palabras contiene.</p>
          <p><strong>💡 Limitaciones observadas:</strong> la precisión puede variar por ruido de fondo, acento argentino, velocidad de habla, volumen y calidad del micrófono. Whisper suele ser más robusto; SpeechRecognition depende de un servicio externo y de la conexión.</p>
        </div>"""
    except Exception as error:
        return info_box("No se pudieron procesar los audios", str(error))


def _resolver_ruta_txt(archivo_txt):
    """Obtiene una ruta real desde str, Path, FileData o diccionarios de Gradio."""
    if archivo_txt is None:
        return None

    if isinstance(archivo_txt, (str, Path)):
        valor = str(archivo_txt).strip()
        return Path(valor) if valor else None

    if isinstance(archivo_txt, dict):
        for clave in ("path", "name", "orig_name"):
            valor = archivo_txt.get(clave)
            if valor:
                ruta = Path(str(valor))
                if ruta.exists() or clave != "orig_name":
                    return ruta

    for atributo in ("path", "name"):
        valor = getattr(archivo_txt, atributo, None)
        if valor:
            return Path(str(valor))

    return None


def leer_archivo_txt(archivo_txt):
    """Lee el TXT al cambiar el selector y sincroniza estado, vista previa y nombre."""
    ruta = _resolver_ruta_txt(archivo_txt)
    if ruta is None:
        return "", "", "Esperando un archivo .txt.", ""

    try:
        if ruta.suffix.lower() != ".txt":
            return "", "", "El archivo seleccionado no tiene formato TXT.", ""

        if not ruta.exists():
            return "", "", "No se pudo acceder al archivo seleccionado. Volvé a cargarlo.", ""

        try:
            texto = ruta.read_text(encoding="utf-8").strip()
        except UnicodeDecodeError:
            texto = ruta.read_text(encoding="latin-1").strip()

        if not texto:
            return "", "", "El archivo está vacío.", ruta.name

        cantidad_palabras = len(texto.split())
        estado = f"✓ Archivo listo: **{ruta.name}** · {cantidad_palabras} palabras."
        # Primer valor: estado interno. Segundo valor: vista previa visible.
        return texto, texto, estado, ruta.name
    except Exception as error:
        return "", "", f"No se pudo leer el archivo: {error}", ""

def _obtener_texto_txt(texto="", archivo_txt=None):
    """Usa el estado del texto o vuelve a leer el archivo de Gradio de forma segura."""
    texto = str(texto or "").strip()
    if texto:
        return texto

    ruta = _resolver_ruta_txt(archivo_txt)
    if ruta is None or not ruta.exists():
        return ""

    try:
        try:
            return ruta.read_text(encoding="utf-8").strip()
        except UnicodeDecodeError:
            return ruta.read_text(encoding="latin-1").strip()
    except Exception:
        return ""

def analizar_texto_txt(texto, archivo_txt=None):
    """Analiza el TXT y explica los resultados en lenguaje sencillo."""
    texto = _obtener_texto_txt(texto, archivo_txt)
    if not texto:
        return "Primero cargá un archivo TXT con contenido."

    tb = engine.sentimiento_textblob(texto)
    vd = engine.sentimiento_vader(texto)
    tr = engine.sentimiento_transformers(texto)

    sentimiento_tb = tb.get("sentimiento", "No disponible")
    sentimiento_vd = vd.get("sentimiento", "No disponible")
    sentimiento_tr = tr.get("sentimiento", "No disponible")
    polaridad = float(tb.get("polaridad", 0) or 0)
    subjetividad = float(tb.get("subjetividad", 0) or 0)
    compound = float(vd.get("compound", 0) or 0)
    confianza = float(tr.get("confianza", 0) or 0)

    consenso, _ = calcular_consenso_sentimientos(tb, vd, tr)
    nivel_acuerdo, explicacion_acuerdo, porcentaje_acuerdo = describir_acuerdo_modelos(tb, vd, tr)
    conclusion = generar_conclusion_tres_modelos(tb, vd, tr, consenso, nivel_acuerdo)
    explicacion_subjetividad = interpretar_subjetividad(subjetividad)

    _, explicacion_polaridad = clasificar_valor_sentimiento(polaridad)
    _, explicacion_compound = clasificar_valor_sentimiento(compound)

    return f"""### Resultado del análisis

| Método | Sentimiento | Métrica principal |
|---|---|---|
| TextBlob | {sentimiento_tb} | Polaridad: {polaridad:.3f} |
| VADER | {sentimiento_vd} | Compound: {compound:.3f} |
| Transformers | {sentimiento_tr} | Confianza: {confianza * 100:.1f} % |

### ¿Qué significa?

**TextBlob:** clasificó el texto como **{sentimiento_tb.lower()}**. Su polaridad fue **{polaridad:.3f}**, en una escala aproximada de -1 (negativo) a 1 (positivo). {explicacion_polaridad}

**VADER:** clasificó el texto como **{sentimiento_vd.lower()}**. El valor *compound* fue **{compound:.3f}**, que resume la intensidad emocional general del texto. {explicacion_compound}

**Transformers:** clasificó el texto como **{sentimiento_tr.lower()}** con una confianza de **{confianza * 100:.1f} %**. Esa confianza indica qué tan seguro estuvo el modelo de su propia clasificación; no representa una certeza absoluta.

**Subjetividad:** {explicacion_subjetividad}

### Interpretación general

{conclusion}

**Acuerdo entre métodos: {nivel_acuerdo} ({porcentaje_acuerdo:.1f} %).** {explicacion_acuerdo}

> Los métodos pueden producir resultados diferentes porque analizan el lenguaje de maneras distintas. Este resultado es orientativo y no constituye un diagnóstico psicológico.
"""


def limpiar_modulo_txt():
    """Restablece los controles y quita el archivo del contexto del chat."""
    return (
        None, "", "", "Esperando un archivo .txt.",
        gr.update(value=None, visible=False), "", "", "",
        '<div class="file-context-empty">Ningún archivo está vinculado al chat.</div>', ""
    )


def convertir_texto_cargado_a_audio(texto, archivo_txt, idioma, velocidad):
    """Genera o actualiza el audio del TXT con valores seguros para Gradio."""
    texto = _obtener_texto_txt(texto, archivo_txt)
    if not texto:
        return (
            gr.update(value=None, visible=False),
            "Primero seleccioná un archivo TXT con contenido.",
            gr.update(visible=False)
        )

    # Gradio puede enviar None durante el primer render o al actualizar componentes.
    idioma = str(idioma or "Español").strip()
    velocidad = str(velocidad or "Normal").strip()

    idiomas_gtts = {
        "Español": "es",
        "Inglés": "en",
        "Francés": "fr",
        "Portugués": "pt",
        "Alemán": "de"
    }
    codigo_idioma = idiomas_gtts.get(idioma, "es")
    lento = velocidad.casefold() == "lenta"

    try:
        # Se genera directamente para poder mostrar el error real si gTTS falla.
        ruta_audio = str(
            Path(tempfile.gettempdir())
            / f"emotichat_txt_{codigo_idioma}_{int(time.time() * 1000)}.mp3"
        )
        gTTS(text=texto, lang=codigo_idioma, slow=lento).save(ruta_audio)

        if velocidad.casefold() == "rápida":
            audio_base = AudioSegment.from_file(ruta_audio)
            audio_rapido = audio_base.speedup(playback_speed=1.25)
            ruta_rapida = str(
                Path(tempfile.gettempdir())
                / f"emotichat_txt_rapido_{int(time.time() * 1000)}.mp3"
            )
            audio_rapido.export(ruta_rapida, format="mp3")
            ruta_audio = ruta_rapida

        if not Path(ruta_audio).exists() or Path(ruta_audio).stat().st_size == 0:
            raise RuntimeError("el archivo de audio no se creó correctamente")

        return (
            gr.update(value=ruta_audio, visible=True),
            f"✓ Audio listo · {idioma} · velocidad {velocidad.lower()}.",
            gr.update(visible=True)
        )
    except Exception as error:
        return (
            gr.update(value=None, visible=False),
            f"No se pudo generar el audio. Detalle: {error}",
            gr.update(visible=True)
        )



def procesar_txt_al_cargar(archivo_txt, idioma="Español", velocidad="Normal"):
    """Lee el TXT y genera automáticamente el audio para mostrar el reproductor."""
    texto, vista_previa, estado_lectura, nombre_archivo = leer_archivo_txt(archivo_txt)

    if not texto:
        return (
            "", "", estado_lectura, nombre_archivo,
            gr.update(value=None, visible=False),
            gr.update(visible=False)
        )

    actualizacion_audio, estado_audio, seccion_audio = convertir_texto_cargado_a_audio(
        texto, archivo_txt, idioma, velocidad
    )

    cantidad_palabras = len(texto.split())
    estado_completo = (
        f"✓ Archivo cargado: **{nombre_archivo}** · {cantidad_palabras} palabras.  \n"
        f"{estado_audio}"
    )

    return (
        texto,
        vista_previa,
        estado_completo,
        nombre_archivo,
        actualizacion_audio,
        seccion_audio
    )


def preparar_experimento_tts(archivo_txt, idioma="Español", velocidad="Normal"):
    """Carga un TXT y genera una variante para comparar voces de gTTS."""
    texto, _, estado_lectura, nombre_archivo = leer_archivo_txt(archivo_txt)
    if not texto:
        return "", gr.update(value=None, visible=False), estado_lectura

    audio, estado_audio, _ = convertir_texto_cargado_a_audio(
        texto, archivo_txt, idioma, velocidad
    )
    return (
        texto,
        audio,
        f"✓ {nombre_archivo} cargado. {estado_audio}"
    )


def actualizar_experimento_tts(texto, archivo_txt, idioma, velocidad):
    """Regenera la voz al cambiar el idioma o la velocidad."""
    audio, estado_audio, _ = convertir_texto_cargado_a_audio(
        texto, archivo_txt, idioma, velocidad
    )
    return audio, estado_audio

def activar_contexto_txt(texto, archivo_txt, nombre_archivo, historial, nombre, etapa):
    """Vincula el TXT al chat sin pegar todo su contenido como mensaje."""
    texto = _obtener_texto_txt(texto, archivo_txt)
    if not texto:
        return "", "", historial, render_chat(historial or estado_inicial(), nombre), (
            '<div class="file-context-empty">Primero cargá un archivo TXT.</div>'
        ), "Primero cargá un archivo TXT."

    historial = list(historial or estado_inicial())
    etiqueta = nombre_archivo or "archivo.txt"
    historial.append({
        "rol": "bot",
        "texto": (
            f"Ya vinculé <strong>{html.escape(etiqueta)}</strong> a la conversación. "
            "Podés pedirme un resumen, la idea principal, el sentimiento o hacer preguntas sobre su contenido."
        ),
        "hora": hora(),
        "html": True
    })
    indicador = (
        '<div class="file-context-active">'
        '<span>📄 Contexto activo</span>'
        f'<strong>{html.escape(etiqueta)}</strong>'
        '<small>Las próximas preguntas pueden responderse usando este archivo.</small>'
        '</div>'
    )
    # La confirmación se comunica dentro del chat y en el indicador de contexto.
    # No se crea una tarjeta adicional separada del flujo de conversación.
    confirmacion = ""
    return texto, etiqueta, historial, render_chat(historial, nombre), indicador, confirmacion


def _normalizar_consulta_txt(texto):
    texto = unicodedata.normalize("NFD", str(texto or "").lower())
    texto = "".join(c for c in texto if unicodedata.category(c) != "Mn")
    texto = re.sub(r"[^a-z0-9ñ\s]", " ", texto)
    return re.sub(r"\s+", " ", texto).strip()


def _oraciones_txt(texto):
    """Divide el documento sin perder párrafos breves ni frases finales."""
    partes = re.split(r"(?<=[.!?])\s+|\n+", str(texto or ""))
    return [p.strip() for p in partes if len(p.strip()) >= 15]


def _resumen_extractivo_txt(texto, max_oraciones=4):
    """Genera un resumen estable y local seleccionando las oraciones más informativas."""
    oraciones = _oraciones_txt(texto)
    if not oraciones:
        return str(texto or "")[:900].strip()
    if len(oraciones) <= max_oraciones:
        return " ".join(oraciones)

    stop = {
        "para", "como", "pero", "porque", "desde", "hasta", "entre", "sobre",
        "este", "esta", "estos", "estas", "esto", "tambien", "aunque", "donde",
        "cuando", "quien", "cual", "cada", "todo", "toda", "todos", "todas",
        "unos", "unas", "del", "las", "los", "una", "que", "con", "por", "sin",
        "sus", "son", "fue", "ser", "han", "hay", "muy", "mas", "menos"
    }
    palabras_doc = re.findall(r"[a-záéíóúüñ]{3,}", _normalizar_consulta_txt(texto))
    frecuencias = {}
    for palabra in palabras_doc:
        if palabra not in stop:
            frecuencias[palabra] = frecuencias.get(palabra, 0) + 1

    puntuadas = []
    total = len(oraciones)
    for indice, oracion in enumerate(oraciones):
        palabras = re.findall(r"[a-záéíóúüñ]{3,}", _normalizar_consulta_txt(oracion))
        relevantes = [p for p in palabras if p not in stop]
        puntaje = sum(frecuencias.get(p, 0) for p in relevantes) / max(len(relevantes), 1)
        if indice == 0:
            puntaje += 1.5
        if indice < max(2, total // 5):
            puntaje += 0.5
        puntuadas.append((puntaje, indice, oracion))

    elegidas = sorted(sorted(puntuadas, reverse=True)[:max_oraciones], key=lambda x: x[1])
    return " ".join(oracion for _, _, oracion in elegidas)


def _palabras_clave_txt(texto, limite=8):
    stop = {
        "para", "como", "pero", "porque", "desde", "hasta", "entre", "sobre",
        "este", "esta", "estos", "estas", "esto", "tambien", "aunque", "donde",
        "cuando", "quien", "cual", "cada", "todo", "toda", "todos", "todas",
        "unos", "unas", "del", "las", "los", "una", "que", "con", "por", "sin",
        "sus", "son", "fue", "ser", "han", "hay", "muy", "mas", "menos"
    }
    frecuencias = {}
    for palabra in re.findall(r"[a-záéíóúüñ]{4,}", _normalizar_consulta_txt(texto)):
        if palabra not in stop:
            frecuencias[palabra] = frecuencias.get(palabra, 0) + 1
    return [p for p, _ in sorted(frecuencias.items(), key=lambda x: (-x[1], x[0]))[:limite]]


def responder_sobre_archivo(pregunta, texto):
    """Responde sobre el TXT con detección de intención y búsqueda extractiva."""
    pregunta_original = str(pregunta or "").strip()
    pregunta_l = _normalizar_consulta_txt(pregunta_original)
    texto = str(texto or "").strip()

    if not texto:
        return "No hay ningún archivo de texto activo en la conversación."

    oraciones = _oraciones_txt(texto)
    resumen = _resumen_extractivo_txt(texto)

    consultas_generales = [
        "que dice el archivo", "que decia el archivo", "que dice el texto",
        "que decia el texto", "contame el archivo", "contame sobre el archivo",
        "explicame el archivo", "explicame el texto", "de que habla el archivo",
        "que contiene el archivo", "que habia en el archivo", "decime que dice"
    ]
    if any(frase in pregunta_l for frase in consultas_generales):
        return "El archivo explica, en resumen: " + resumen

    if any(x in pregunta_l for x in ["resum", "sintesis", "brevemente", "en pocas palabras"]):
        return "Resumen del archivo: " + resumen

    if any(x in pregunta_l for x in ["idea principal", "tema principal", "de que trata", "tema del archivo", "asunto principal"]):
        palabras = _palabras_clave_txt(texto, 5)
        tema = ", ".join(palabras) if palabras else "el contenido presentado"
        primera = oraciones[0] if oraciones else resumen
        return f"La idea principal se relaciona con {tema}. En el documento se plantea: {primera}"

    if any(x in pregunta_l for x in ["palabras clave", "conceptos clave", "temas clave"]):
        palabras = _palabras_clave_txt(texto, 8)
        return "Las palabras o conceptos más frecuentes del archivo son: " + ", ".join(palabras) + "."

    if any(x in pregunta_l for x in ["sentimiento", "emocion", "tono", "como se siente"]):
        tb = engine.sentimiento_textblob(texto)
        vd = engine.sentimiento_vader(texto)
        tr = engine.sentimiento_transformers(texto)
        consenso, _ = calcular_consenso_sentimientos(tb, vd, tr)
        nivel, explicacion, porcentaje = describir_acuerdo_modelos(tb, vd, tr)
        return (
            f"El análisis general del archivo es {consenso.lower()}. "
            f"TextBlob indicó {tb.get('sentimiento', 'no disponible')}, "
            f"VADER {vd.get('sentimiento', 'no disponible')} y Transformers "
            f"{tr.get('sentimiento', 'no disponible')}. "
            f"El acuerdo entre métodos es {nivel.lower()} ({porcentaje:.1f} %). {explicacion}"
        )

    if any(x in pregunta_l for x in ["quien aparece", "quienes aparecen", "nombres", "personas menciona", "persona menciona"]):
        nombres = re.findall(r"(?<![.!?]\s)\b[A-ZÁÉÍÓÚÜÑ][a-záéíóúüñ]+(?:\s+[A-ZÁÉÍÓÚÜÑ][a-záéíóúüñ]+)?", texto)
        nombres_unicos = []
        for nombre in nombres:
            if nombre not in nombres_unicos:
                nombres_unicos.append(nombre)
        if nombres_unicos:
            return "En el archivo aparecen estos nombres propios: " + ", ".join(nombres_unicos[:12]) + "."
        return "No pude identificar nombres propios con suficiente claridad en el archivo."

    stop_consulta = {
        "sobre", "archivo", "texto", "cual", "como", "este", "esta", "para",
        "dice", "decia", "dime", "decime", "pregunta", "quiero", "saber", "algo",
        "contenido", "documento", "menciona", "habla", "explica"
    }
    palabras_pregunta = [
        p for p in re.findall(r"[a-záéíóúüñ]{3,}", pregunta_l)
        if p not in stop_consulta
    ]

    puntuadas = []
    for indice, oracion in enumerate(oraciones):
        normalizada = _normalizar_consulta_txt(oracion)
        palabras_oracion = set(re.findall(r"[a-záéíóúüñ]{3,}", normalizada))
        coincidencias = sum(1 for p in palabras_pregunta if p in palabras_oracion)
        similitud = difflib.SequenceMatcher(None, pregunta_l, normalizada).ratio()
        puntaje = coincidencias * 3 + similitud
        if coincidencias or similitud >= 0.28:
            puntuadas.append((puntaje, indice, oracion))

    relevantes = sorted(puntuadas, reverse=True)[:3]
    if relevantes:
        relevantes = [o for _, _, o in sorted(relevantes, key=lambda x: x[1])]
        return "Según el archivo: " + " ".join(relevantes)

    return (
        "No encontré una frase específica que responda exactamente eso. "
        "De todos modos, el contenido general del archivo es: " + resumen
    )


def ejecutar_pipeline_desde_audio(ultima_ruta_audio=None):
    ruta_audio = ultima_ruta_audio

    if not ruta_audio:
        return (
            "",
            "",
            "",
            gr.update(value=None, visible=False),
            "Primero enviá o grabá un audio en Conversación, dentro de la pestaña Hablar."
        )

    try:
        resultado_transcripcion = engine.transcribir_whisper(ruta_audio)

        if not resultado_transcripcion.get("ok"):
            return (
                "",
                "",
                "",
                gr.update(value=None, visible=False),
                resultado_transcripcion.get(
                    "error",
                    "No se pudo transcribir el audio."
                )
            )

        texto_transcrito = resultado_transcripcion.get("texto", "").strip()

        if not texto_transcrito:
            return (
                "",
                "",
                "",
                gr.update(value=None, visible=False),
                "La transcripción quedó vacía."
            )

        datos_pipeline = engine.generar_resumen_pipeline(texto_transcrito)
        sentimiento = datos_pipeline.get(
            "sentimiento_general",
            "No disponible"
        )
        resumen = datos_pipeline.get("resumen", "")

        ruta_resumen_audio = engine.texto_a_voz_configurable(
            texto=resumen,
            idioma="es",
            lento=False
        )

        if not ruta_resumen_audio:
            return (
                texto_transcrito,
                sentimiento,
                resumen,
                gr.update(value=None, visible=False),
                "El análisis se completó, pero no se pudo generar el audio final."
            )

        tiempo = resultado_transcripcion.get("tiempo", 0)
        return (
            texto_transcrito,
            sentimiento,
            resumen,
            gr.update(value=ruta_resumen_audio, visible=True),
            f"Pipeline completado. Tiempo de transcripción: {tiempo} segundos."
        )

    except Exception as error:
        return (
            "",
            "",
            "",
            gr.update(value=None, visible=False),
            f"Error durante el pipeline: {error}"
        )


def sidebar_conclusiones():
    return sidebar_info(
        "Conclusiones",
        "Accede al informe final con los resultados obtenidos y las conclusiones del procesamiento."
    )



def panel_conclusiones(historial=None):
    contexto = obtener_contexto_usuario(historial or [])
    if contexto:
        tb = engine.sentimiento_textblob(contexto)
        vd = engine.sentimiento_vader(contexto)
        tr = engine.sentimiento_transformers(contexto)
        consenso, confianza = calcular_consenso_sentimientos(tb, vd, tr)
        traduccion = engine.traduccion_encadenada(contexto)
        retro = analizar_retrotraduccion(traduccion.get("original", contexto), traduccion.get("final", contexto))
        idioma = NOMBRES_IDIOMAS.get(traduccion.get("idioma_origen", "es"), traduccion.get("idioma_origen", "es"))
    else:
        tb={"sentimiento":"Sin datos","polaridad":0,"subjetividad":0}
        vd={"sentimiento":"Sin datos","compound":0}
        tr={"sentimiento":"Sin datos","confianza":0}
        consenso, confianza, idioma = "Sin datos", 0, "Sin datos"
        retro={"similitud":0,"explicacion":"Escribí mensajes en el chat para generar resultados aplicados a la conversación."}

    teoria = [
        ("Whisper", "Convierte audio en texto mediante un modelo Transformer multilingüe.", "Es robusto frente a ruido y acentos, aunque requiere más recursos."),
        ("SpeechRecognition", "Permite utilizar motores de reconocimiento de voz desde Python.", "Es rápido y simple, pero más sensible al ruido y a la conexión."),
        ("TextBlob", "Calcula polaridad y subjetividad mediante un enfoque léxico.", "Es fácil de interpretar, pero comprende de forma limitada el contexto y la ironía."),
        ("VADER", "Mide intensidad emocional con puntajes negativo, neutro, positivo y compound.", "Funciona bien con textos breves, aunque su léxico original está orientado al inglés."),
        ("Transformers", "Analiza el contexto completo y devuelve una etiqueta con confianza.", "Aporta comprensión contextual, pero consume más recursos y puede heredar sesgos."),
        ("GoogleTranslator", "Realiza traducción automática y retrotraducción.", "Permite observar cambios de significado, modismos o intensidad emocional."),
        ("SpeechBrain", "Analiza características acústicas y emocionales de la voz.", "Complementa el texto, pero depende del idioma, acento y ruido."),
        ("gTTS", "Convierte texto en audio mediante síntesis de voz.", "Completa el pipeline, aunque depende de Internet.")
    ]
    teoria_html = "".join(
        f"<details class='tech-detail'><summary>{n}</summary><div><p><strong>Qué hace:</strong> {q}</p><p><strong>Alcance y limitación:</strong> {l}</p></div></details>"
        for n,q,l in teoria
    )
    contenido = (
        f'<div class="result-grid executive-grid">'
        f'<div class="result-card"><span>Estado emocional</span><strong>{html.escape(str(consenso))}</strong></div>'
        f'<div class="result-card"><span>Confianza general</span><strong>{confianza} %</strong></div>'
        f'<div class="result-card"><span>Idioma</span><strong>{html.escape(str(idioma))}</strong></div>'
        f'<div class="result-card"><span>Retrotraducción</span><strong>{retro["similitud"]} %</strong></div></div>'
        '<div class="module-card wide-card"><h3>Resultados aplicados a la conversación</h3><div class="model-results-grid">'
        f'<div><span>TextBlob</span><strong>{tb.get("sentimiento","—")}</strong><small>Polaridad {tb.get("polaridad",0)} · Subjetividad {tb.get("subjetividad",0)}</small></div>'
        f'<div><span>VADER</span><strong>{vd.get("sentimiento","—")}</strong><small>Compound {vd.get("compound",0)}</small></div>'
        f'<div><span>Transformers</span><strong>{tr.get("sentimiento","—")}</strong><small>Confianza {round(float(tr.get("confianza",0))*100,1)} %</small></div></div></div>'
        f'<div class="interpretation-card"><div class="interpretation-header"><span>Interpretación integrada</span><b>{confianza} % de confianza</b></div>'
        '<p>El consenso compara TextBlob, VADER y Transformers. La coincidencia entre métodos aumenta la solidez del resultado, pero no representa una evaluación clínica.</p>'
        f'<p><strong>Traducción:</strong> {html.escape(retro["explicacion"])}</p></div>'
        f'<div class="module-card wide-card"><h3>Tecnologías y fundamentos</h3><p class="section-intro">Se presentan en secciones desplegables para facilitar la lectura.</p><div class="tech-accordion">{teoria_html}</div></div>'
        '<div class="comparison-strip"><div><strong>Transcripción</strong><span>Whisper aporta robustez; SpeechRecognition rapidez.</span></div><div><strong>Sentimientos</strong><span>Tres enfoques reducen la dependencia de un único modelo.</span></div><div><strong>Traducción</strong><span>La retrotraducción permite detectar cambios de matiz.</span></div><div><strong>Alcance</strong><span>Prototipo de demostración; no reemplaza asistencia profesional.</span></div></div>'
        '<div class="final-conclusion-card" style="background:#243b5a;color:#ffffff;"><span style="color:#ffffff;font-weight:700;">Conclusión general</span><p style="color:#f4f7fb;margin:8px 0 0;line-height:1.6;">EmotiChat integra voz, traducción, análisis emocional y síntesis en un único flujo. Su fortaleza es comparar tecnologías, explicar resultados y mostrar limitaciones de manera responsable.</p></div>'
        '<div class="info-box"><h4>Informe PDF</h4><p>Desde esta sección podés generar un PDF breve con las conclusiones de la sesión.</p></div>'
    )
    return panel_modulo("Conclusiones", "Resultados, interpretación y fundamentos técnicos", contenido)


def mostrar_conclusiones(historial):
    return sidebar_conclusiones(), panel_conclusiones(historial), gr.update(visible=True)


def generar_pdf_descargable(historial, nombre):
    """
    Genera un PDF breve con solamente las conclusiones de la sesión:
    estado emocional, resumen y recomendaciones.
    No incluye el chat ni las tablas técnicas.
    """
    if not historial:
        return gr.update(value=None, visible=False)

    contexto = obtener_contexto_usuario(historial, limite=3)
    if not contexto:
        return gr.update(value=None, visible=False)

    # Se reutilizan los tres modelos ya presentes en EmotiChat.
    tb = engine.sentimiento_textblob(contexto)
    vd = engine.sentimiento_vader(contexto)
    tr = engine.sentimiento_transformers(contexto)

    consenso, porcentaje_confianza = calcular_consenso_sentimientos(tb, vd, tr)
    nivel_confianza, explicacion_confianza, _ = describir_acuerdo_modelos(tb, vd, tr)
    conclusion = generar_conclusion_tres_modelos(
        tb, vd, tr, consenso, nivel_confianza
    )
    recomendacion = generar_recomendacion_consenso(consenso)

    ruta_pdf = BASE_DIR / f"conclusiones_emotichat_{int(time.time())}.pdf"

    violeta = colors.HexColor("#6457E8")
    violeta_claro = colors.HexColor("#EEEAFE")
    texto_oscuro = colors.HexColor("#253046")
    texto_suave = colors.HexColor("#667085")
    borde = colors.HexColor("#D9DDF0")
    blanco = colors.white

    doc = SimpleDocTemplate(
        str(ruta_pdf),
        pagesize=A4,
        rightMargin=44,
        leftMargin=44,
        topMargin=38,
        bottomMargin=44,
        title="Conclusiones de la sesión - EmotiChat",
        author="Paola F. Dueña"
    )

    styles = getSampleStyleSheet()
    estilo_titulo = ParagraphStyle(
        "TituloEmotiChat",
        parent=styles["Title"],
        fontName="Helvetica-Bold",
        fontSize=21,
        leading=25,
        textColor=blanco,
        alignment=TA_CENTER
    )
    estilo_subtitulo = ParagraphStyle(
        "SubtituloEmotiChat",
        parent=styles["Normal"],
        fontName="Helvetica",
        fontSize=9.5,
        leading=13,
        textColor=colors.HexColor("#EDEBFF"),
        alignment=TA_CENTER
    )
    estilo_seccion = ParagraphStyle(
        "SeccionEmotiChat",
        parent=styles["Heading2"],
        fontName="Helvetica-Bold",
        fontSize=13,
        leading=17,
        textColor=violeta,
        spaceBefore=10,
        spaceAfter=7
    )
    estilo_normal = ParagraphStyle(
        "NormalEmotiChat",
        parent=styles["Normal"],
        fontName="Helvetica",
        fontSize=10,
        leading=15,
        textColor=texto_oscuro
    )
    estilo_pequeno = ParagraphStyle(
        "PequenoEmotiChat",
        parent=styles["Normal"],
        fontName="Helvetica",
        fontSize=8.5,
        leading=12,
        textColor=texto_suave
    )
    estilo_destacado = ParagraphStyle(
        "DestacadoEmotiChat",
        parent=estilo_normal,
        backColor=violeta_claro,
        borderColor=borde,
        borderWidth=0.7,
        borderPadding=10,
        leading=15
    )

    elementos = []

    logo = None
    try:
        logo_url = (
            "https://raw.githubusercontent.com/"
            "PaoRioColorado/EmotiChat/main/Logo_EmotiChat.png"
        )
        respuesta_logo = requests.get(logo_url, timeout=12)
        respuesta_logo.raise_for_status()
        logo = Image(io.BytesIO(respuesta_logo.content), width=72, height=72)
    except Exception:
        logo = Paragraph("EC", estilo_titulo)

    encabezado = Table(
        [[logo, [
            Paragraph("EmotiChat", estilo_titulo),
            Paragraph("Conclusiones de la sesión", estilo_subtitulo)
        ]]],
        colWidths=[90, 405],
        rowHeights=[84]
    )
    encabezado.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, -1), violeta),
        ("VALIGN", (0, 0), (-1, -1), "MIDDLE"),
        ("ALIGN", (0, 0), (0, 0), "CENTER"),
        ("LEFTPADDING", (0, 0), (-1, -1), 12),
        ("RIGHTPADDING", (0, 0), (-1, -1), 12),
        ("TOPPADDING", (0, 0), (-1, -1), 7),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 7),
        ("BOX", (0, 0), (-1, -1), 0.8, violeta),
    ]))
    elementos.append(encabezado)
    elementos.append(Spacer(1, 14))

    nombre_seguro = html.escape(nombre or "Sin nombre")
    fecha = datetime.now(
        ZoneInfo("America/Argentina/Buenos_Aires")
    ).strftime("%d/%m/%Y %H:%M")

    datos = Table(
        [[
            Paragraph("<b>Persona</b>", estilo_pequeno),
            Paragraph(nombre_seguro, estilo_normal),
            Paragraph("<b>Fecha</b>", estilo_pequeno),
            Paragraph(fecha, estilo_normal)
        ]],
        colWidths=[55, 175, 45, 220]
    )
    datos.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, -1), colors.HexColor("#F7F5FF")),
        ("BOX", (0, 0), (-1, -1), 0.7, borde),
        ("VALIGN", (0, 0), (-1, -1), "MIDDLE"),
        ("LEFTPADDING", (0, 0), (-1, -1), 8),
        ("RIGHTPADDING", (0, 0), (-1, -1), 8),
        ("TOPPADDING", (0, 0), (-1, -1), 7),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 7),
    ]))
    elementos.append(datos)
    elementos.append(Spacer(1, 14))

    elementos.append(Paragraph("Estado emocional", estilo_seccion))
    elementos.append(Paragraph(
        f"<b>{html.escape(consenso)}</b><br/>"
        f"Confianza estimada: {porcentaje_confianza}% "
        f"({html.escape(nivel_confianza.lower())}).<br/>"
        f"{html.escape(explicacion_confianza)}",
        estilo_destacado
    ))

    elementos.append(Paragraph("Resumen", estilo_seccion))
    elementos.append(Paragraph(html.escape(conclusion), estilo_normal))

    elementos.append(Paragraph("Recomendaciones", estilo_seccion))
    elementos.append(Paragraph(html.escape(recomendacion), estilo_normal))

    elementos.append(Spacer(1, 16))
    elementos.append(Paragraph(
        "Este resultado es orientativo y surge del análisis automático de "
        "TextBlob, VADER y Transformers sobre los últimos tres mensajes. "
        "No constituye un diagnóstico psicológico, psiquiátrico ni clínico.",
        estilo_destacado
    ))

    def agregar_pie_pagina(canvas, documento):
        canvas.saveState()
        ancho, _ = A4
        canvas.setStrokeColor(borde)
        canvas.setLineWidth(0.5)
        canvas.line(44, 30, ancho - 44, 30)
        canvas.setFont("Helvetica", 8)
        canvas.setFillColor(texto_suave)
        canvas.drawString(
            44, 18, "EmotiChat · Desarrollado por Paola F. Dueña"
        )
        canvas.drawRightString(
            ancho - 44, 18, f"Página {canvas.getPageNumber()}"
        )
        canvas.restoreState()

    doc.build(
        elementos,
        onFirstPage=agregar_pie_pagina,
        onLaterPages=agregar_pie_pagina
    )
    return gr.update(value=str(ruta_pdf), visible=True)


def mostrar_conversacion(nombre, etapa):
    return sidebar_conversacion(nombre, etapa), panel_inicial() if etapa == "nombre" else panel_sesion(nombre)


def mostrar_audio(historial):
    return sidebar_audio(), panel_audio(historial)


def mostrar_traduccion(historial):
    return sidebar_traduccion(), panel_traduccion_menu(historial)


def mostrar_sentimientos(historial):
    return sidebar_sentimientos(), panel_sentimientos_menu(historial)




In [ ]:
# ==============================================================================
# 8. CONTROLADORES Y FLUJOS DE INTERACCIÓN


In [ ]:


def normalizar_para_similitud(texto):
    """
    Normaliza un texto antes de compararlo:
    - pasa a minúsculas;
    - elimina tildes;
    - elimina signos;
    - unifica espacios.
    """
    texto = str(texto or "").strip().lower()
    texto = unicodedata.normalize("NFD", texto)
    texto = "".join(
        caracter for caracter in texto
        if unicodedata.category(caracter) != "Mn"
    )
    texto = re.sub(r"[^a-z0-9ñ\s]", " ", texto)
    return re.sub(r"\s+", " ", texto).strip()


def calcular_porcentaje_similitud(texto_referencia, texto_obtenido):
    """
    Compara las secuencias de palabras y devuelve un porcentaje entre 0 y 100.
    """
    referencia = normalizar_para_similitud(texto_referencia).split()
    obtenido = normalizar_para_similitud(texto_obtenido).split()

    if not referencia or not obtenido:
        return 0.0

    valor = difflib.SequenceMatcher(
        None,
        referencia,
        obtenido
    ).ratio()

    return round(valor * 100, 1)


def interpretar_similitud(porcentaje):
    if porcentaje >= 95:
        return "Muy alta", "La transcripción conserva casi todo el texto de referencia."
    if porcentaje >= 85:
        return "Alta", "La transcripción presenta pocas diferencias."
    if porcentaje >= 70:
        return "Media", "Se observan varias diferencias u omisiones."
    if porcentaje > 0:
        return "Baja", "La transcripción difiere considerablemente del texto de referencia."
    return "No disponible", "No fue posible calcular la similitud."


def estado_metodo_transcripcion(resultado):
    return "Completado" if resultado.get("ok") else "No disponible"


def texto_metodo_transcripcion(resultado):
    if resultado.get("ok"):
        return resultado.get("texto", "")
    return resultado.get("error", "No se obtuvo un resultado.")


def panel_comparacion_transcripcion(comparacion, texto_referencia):
    whisper_resultado = comparacion["whisper"]
    speech_resultado = comparacion["speech_recognition"]

    whisper_texto = texto_metodo_transcripcion(whisper_resultado)
    speech_texto = texto_metodo_transcripcion(speech_resultado)

    similitud_whisper = (
        calcular_porcentaje_similitud(texto_referencia, whisper_texto)
        if whisper_resultado.get("ok") else 0
    )
    similitud_speech = (
        calcular_porcentaje_similitud(texto_referencia, speech_texto)
        if speech_resultado.get("ok") else 0
    )

    nivel_whisper, observacion_whisper = interpretar_similitud(similitud_whisper)
    nivel_speech, observacion_speech = interpretar_similitud(similitud_speech)

    filas = [
        (
            "Whisper",
            estado_metodo_transcripcion(whisper_resultado),
            f'{whisper_resultado.get("tiempo", 0)} s',
            whisper_texto,
            f"{similitud_whisper:.1f} %" if whisper_resultado.get("ok") else "—",
            nivel_whisper,
            observacion_whisper
        ),
        (
            "SpeechRecognition",
            estado_metodo_transcripcion(speech_resultado),
            f'{speech_resultado.get("tiempo", 0)} s',
            speech_texto,
            f"{similitud_speech:.1f} %" if speech_resultado.get("ok") else "—",
            nivel_speech,
            observacion_speech
        )
    ]

    cuerpo = "".join(
        f"""
        <tr>
            <td><strong>{html.escape(metodo)}</strong></td>
            <td>{html.escape(estado)}</td>
            <td>{html.escape(tiempo)}</td>
            <td>{html.escape(texto)}</td>
            <td><strong>{html.escape(similitud)}</strong></td>
            <td>{html.escape(nivel)}</td>
            <td>{html.escape(observacion)}</td>
        </tr>
        """
        for metodo, estado, tiempo, texto, similitud, nivel, observacion in filas
    )

    tabla = f"""
    <div class="comparison-table-scroll">
        <table class="metric-table transcription-comparison-table">
            <thead>
                <tr>
                    <th>Método</th>
                    <th>Estado</th>
                    <th>Tiempo</th>
                    <th>Transcripción obtenida</th>
                    <th>Similitud</th>
                    <th>Nivel</th>
                    <th>Observaciones</th>
                </tr>
            </thead>
            <tbody>{cuerpo}</tbody>
        </table>
    </div>
    """

    resultados_validos = []
    if whisper_resultado.get("ok"):
        resultados_validos.append(("Whisper", similitud_whisper))
    if speech_resultado.get("ok"):
        resultados_validos.append(("SpeechRecognition", similitud_speech))

    if resultados_validos:
        mejor_metodo, mejor_similitud = max(
            resultados_validos,
            key=lambda elemento: elemento[1]
        )
        conclusion = (
            f"{mejor_metodo} obtuvo el porcentaje de similitud más alto "
            f"({mejor_similitud:.1f} %) para este audio."
        )
    else:
        conclusion = "No se pudo obtener una transcripción válida para comparar."

    contenido = f"""
        <div class="module-card wide-card reference-card">
            <h3>Texto de referencia</h3>
            <p>{html.escape(texto_referencia)}</p>
        </div>

        <div class="module-card wide-card">
            <h3>Comparación de métodos</h3>
            {tabla}
        </div>

        <div class="module-card wide-card">
            <h3>Explicación</h3>
            <p>
                El porcentaje de similitud indica qué tan parecida es cada
                transcripción al texto de referencia escrito por la persona.
                Un resultado cercano al 100 % significa que se reconocieron
                correctamente la mayoría de las palabras.
            </p>
            <p><strong>Resultado:</strong> {html.escape(conclusion)}</p>
        </div>

        <div class="tech-pill">
            Procesado con: OpenAI Whisper • SpeechRecognition • difflib
        </div>
    """

    return panel_modulo(
        "Procesamiento de voz",
        "Comparación de transcripción con porcentaje de similitud",
        contenido
    )


def comparar_audio_con_referencia(
    audio_grabado,
    audio_archivo,
    texto_referencia
):
    """Compara un audio grabado o seleccionado con ambos métodos."""
    ruta_audio = audio_archivo or audio_grabado
    referencia = str(texto_referencia or "").strip()

    if not ruta_audio:
        return panel_modulo(
            "Procesamiento de voz",
            "Comparación de transcripción",
            info_box(
                "Falta el audio",
                "Grabá un mensaje o subí un archivo antes de iniciar la comparación."
            )
        )

    if not referencia:
        return panel_modulo(
            "Procesamiento de voz",
            "Comparación de transcripción",
            info_box(
                "Falta el texto de referencia",
                "Escribí exactamente lo que se dijo en el audio para poder calcular la similitud."
            )
        )

    try:
        comparacion = engine.comparar_transcripciones(ruta_audio)
        return panel_comparacion_transcripcion(
            comparacion,
            referencia
        )
    except Exception as error:
        return panel_modulo(
            "Procesamiento de voz",
            "Comparación de transcripción",
            info_box(
                "No se pudo completar la comparación",
                html.escape(str(error))
            )
        )


def mostrar_comparador_transcripcion():
    return gr.update(visible=True)


def ocultar_comparador_transcripcion():
    return gr.update(visible=False)


def panel_error_modo_voz(mensaje_error):
    tarjetas = (
        card_html(
            "⚠",
            "Modo conversación por voz",
            "No se pudo procesar",
            html.escape(mensaje_error),
            "orange-card"
        )
        +
        info_box(
            "Cómo intentarlo nuevamente",
            "Tocá el micrófono, hablá con claridad y detené la grabación. "
            "También podés subir un archivo de audio."
        )
    )

    return panel_modulo(
        "Conversación por voz",
        "Grabación, transcripción y respuesta hablada",
        tarjetas
    )


def esperar_audio_guardado(ruta_audio):
    """
    Espera a que Gradio termine de crear el archivo antes de enviarlo
    a Whisper. Realiza varios intentos porque algunos navegadores tardan
    un poco más al detener la grabación.
    """
    print(f"[AUDIO] Valor recibido al detener: {ruta_audio!r}")

    if not ruta_audio:
        print("[AUDIO] No llegó ninguna ruta desde el navegador.")
        return None

    ruta = Path(str(ruta_audio))

    for intento in range(12):
        if ruta.exists():
            tamaño = ruta.stat().st_size
            print(
                f"[AUDIO] Intento {intento + 1}: "
                f"archivo encontrado, tamaño={tamaño} bytes"
            )
            if tamaño > 1000:
                return str(ruta)

        time.sleep(0.25)

    print(
        f"[AUDIO] El archivo no quedó disponible o quedó vacío: {ruta}"
    )
    return None


def identificador_audio(ruta_audio):
    """Crea una firma breve para evitar procesar dos veces el mismo archivo."""
    try:
        ruta = Path(ruta_audio)
        datos = f"{ruta.resolve()}|{ruta.stat().st_size}|{ruta.stat().st_mtime_ns}"
        return hashlib.sha1(datos.encode("utf-8")).hexdigest()
    except Exception:
        return str(ruta_audio or "")


def panel_transcripcion_multimodal(resultado, resultado_voz, comparacion):
    texto = resultado.get("texto", "")
    tiempo_whisper = resultado.get("tiempo", 0)

    if resultado_voz.get("ok"):
        voz_html = f"""
        <div class="module-card">
            <h3>Emoción estimada en la voz</h3>
            <p><strong>{html.escape(resultado_voz["emocion"])}</strong></p>
            <p>Confianza: {resultado_voz["confianza"] * 100:.1f} %</p>
            <p>Tiempo: {resultado_voz.get("tiempo", 0):.2f} s</p>
        </div>
        """
    else:
        voz_html = f"""
        <div class="module-card">
            <h3>Emoción estimada en la voz</h3>
            <p>No disponible.</p>
            <p class="muted">{html.escape(resultado_voz.get("error", ""))}</p>
        </div>
        """

    contenido = f"""
        {card_html(
            icono_svg("microphone"),
            "Audio procesado",
            "Español",
            html.escape(texto),
            "blue-card"
        )}

        <div class="module-grid">
            <div class="module-card">
                <h3>Sentimiento del texto</h3>
                <p><strong>{html.escape(comparacion["sentimiento_texto"])}</strong></p>
                <p>Confianza: {comparacion["confianza_texto"] * 100:.1f} %</p>
            </div>

            {voz_html}
        </div>

        <div class="module-card wide-card">
            <h3>Análisis multimodal</h3>
            <p>{html.escape(comparacion["conclusion"])}</p>
            <p>
                Whisper: {tiempo_whisper:.2f} s ·
                SpeechBrain: {resultado_voz.get("tiempo", 0):.2f} s
            </p>
        </div>

        <div class="info-box">
            <strong>Resultado experimental</strong>
            <p>
                SpeechBrain estima características emocionales acústicas.
                No reemplaza una evaluación profesional y puede verse afectado
                por idioma, acento, ruido, micrófono y estilo de habla.
            </p>
        </div>
    """

    return panel_modulo(
        "Procesamiento de voz",
        "Transcripción y análisis emocional multimodal",
        contenido
    )


def panel_transcripcion_rapida(resultado):
    if resultado.get("ok"):
        tarjetas = (
            card_html(
                icono_svg("microphone"),
                "Audio transcripto",
                "Español",
                html.escape(resultado.get("texto", "")),
                "blue-card"
            )
            +
            info_box(
                "Modo conversación",
                f"Whisper BASE completó la transcripción en "
                f"{resultado.get('tiempo', 0)} segundos. "
                "Las comparaciones y análisis completos se ejecutan "
                "desde sus módulos para mantener fluido el chat."
            )
        )
        return panel_modulo(
            "Conversación por voz",
            "Transcripción rápida para continuar el chat",
            tarjetas
        )

    return panel_error_modo_voz(
        resultado.get("error", "No se pudo transcribir el audio.")
    )


def procesar_modo_voz(
    ruta_audio,
    historial,
    nombre,
    etapa,
    ultimo_audio_procesado,
    ultima_ruta_audio
):
    """
    Flujo liviano para conversar:
    audio → Whisper BASE → chat → gTTS.

    SpeechRecognition, traducciones y análisis se mantienen disponibles
    en sus módulos, pero no se ejecutan en cada turno de voz.
    """
    if historial is None:
        historial = estado_inicial()

    if not ruta_audio:
        yield (
            render_topbar(nombre),
            render_chat(historial, nombre),
            gr.update(),
            historial,
            nombre,
            etapa,
            gr.update(),
            gr.update(),
            gr.update(),
            ultimo_audio_procesado,
            ultima_ruta_audio
        )
        return

    firma_actual = identificador_audio(ruta_audio)

    # Protección contra eventos duplicados del navegador.
    if firma_actual and firma_actual == ultimo_audio_procesado:
        yield (
            render_topbar(nombre),
            render_chat(historial, nombre),
            gr.update(),
            historial,
            nombre,
            etapa,
            gr.update(),
            gr.update(),
            gr.update(value=None),
            ultimo_audio_procesado,
            ultima_ruta_audio
        )
        return

    resultado_whisper = engine.transcribir_whisper(ruta_audio)
    texto_transcripto = (
        resultado_whisper.get("texto", "").strip()
        if resultado_whisper.get("ok")
        else ""
    )

    # El chat evita SpeechBrain para responder rápido.
    # El análisis multimodal completo sigue disponible en el módulo técnico.
    panel_voz = panel_transcripcion_rapida(resultado_whisper)

    if not texto_transcripto:
        yield (
            render_topbar(nombre),
            render_chat(historial, nombre),
            gr.update(value=""),
            historial,
            nombre,
            etapa,
            panel_voz,
            gr.update(value=None, visible=False),
            gr.update(value=None),
            firma_actual,
            str(ruta_audio)
        )
        return

    # Muestra la transcripción antes de generar la respuesta.
    yield (
        render_topbar(nombre),
        render_chat(historial, nombre),
        gr.update(value=texto_transcripto),
        historial,
        nombre,
        etapa,
        panel_voz,
        gr.update(value=None, visible=False),
        gr.update(),
        firma_actual,
        str(ruta_audio)
    )

    nombre_flujo = nombre
    etapa_flujo = etapa
    if (
        etapa == "nombre"
        and not extraer_nombre(texto_transcripto)
        and parece_mensaje_conversacional(texto_transcripto)
    ):
        nombre_flujo = "Invitad@"
        etapa_flujo = "chat"

    resultados_chat = list(
        procesar(
            texto_transcripto,
            historial,
            nombre_flujo,
            etapa_flujo
        )
    )

    for indice, salida_original in enumerate(resultados_chat):
        salida = list(salida_original)

        if indice == len(resultados_chat) - 1:
            historial_final = salida[3]
            ruta_respuesta = None

            if historial_final:
                for item in reversed(historial_final):
                    if item.get("rol") == "bot":
                        # La voz reproduce solamente la respuesta principal.
                        # Los recursos visuales y sus iconos permanecen en pantalla.
                        texto_principal = item.get("texto", "").split(
                            '<div class="inline-resources-divider">', 1
                        )[0]
                        texto_hablado = html.unescape(
                            re.sub(r"<[^>]+>", " ", texto_principal)
                        )
                        texto_hablado = re.sub(
                            r"[\U00010000-\U0010ffff\u2600-\u27BF]",
                            " ",
                            texto_hablado
                        )
                        texto_hablado = re.sub(r"\s+", " ", texto_hablado).strip()
                        ruta_respuesta = engine.texto_a_voz(texto_hablado)
                        break

            salida[7] = gr.update(
                value=ruta_respuesta,
                visible=bool(ruta_respuesta)
            )

            # Limpia la grabación anterior solamente cuando todo terminó.
            salida.append(gr.update(value=None))
        else:
            salida.append(gr.update())

        salida.append(firma_actual)
        salida.append(str(ruta_audio))
        yield tuple(salida)



# ==============================================================================

def procesar(mensaje, historial, nombre, etapa, contexto_txt="", nombre_archivo_txt=""):
    if historial is None:
        historial = estado_inicial()

    if not mensaje or not mensaje.strip():
        historial.append({
            "rol": "bot",
            "texto": "Necesito que escribas algo para continuar.",
            "hora": hora()
        })
        panel = panel_inicial() if etapa == "nombre" else panel_sesion(nombre)

        yield (
            render_topbar(nombre),
            render_chat(historial, nombre),
            gr.update(
                value="",
                placeholder="Escribí tu nombre o contame cómo te sentís..." if etapa == "nombre" else "Escribe cómo te sientes hoy..."
            ),
            historial,
            nombre,
            etapa,
            panel,
            None
        )
        return

    mensaje = mensaje.strip()

    # ETAPA NOMBRE: reconoce el nombre aunque venga acompañado
    # por una descripción emocional, por ejemplo:
    # "Hola, soy Paola y estoy bien".
    if etapa == "nombre" and not contexto_txt:
        valido, nombre_detectado, error = validar_nombre(mensaje)

        if not valido and parece_mensaje_conversacional(mensaje):
            # La persona comenzó hablando de cómo se siente. Se continúa como
            # invitada y no se vuelve a interrumpir la conversación por el nombre.
            nombre = "Invitad@"
            etapa = "chat"

        elif not valido:
            historial.append({"rol": "bot", "texto": error, "hora": hora()})
            yield (
                render_topbar(nombre),
                render_chat(historial, nombre),
                gr.update(
                    value="",
                    placeholder="Escribí tu nombre o contame cómo te sentís..."
                ),
                historial,
                nombre,
                etapa,
                panel_inicial(),
                None
            )
            return

        else:
            nombre = nombre_detectado
            etapa = "chat"
            resto_mensaje = extraer_mensaje_despues_del_nombre(mensaje, nombre)

            historial.append({
                "rol": "bot",
                "texto": (
                    f"Mucho gusto, {nombre}."
                    if resto_mensaje
                    else f"Mucho gusto, {nombre}.\n\n¿Cómo te sentís hoy?"
                ),
                "hora": hora()
            })

            if not resto_mensaje:
                yield (
                    render_topbar(nombre),
                    render_chat(historial, nombre),
                    gr.update(value="", placeholder="Escribe cómo te sientes hoy..."),
                    historial,
                    nombre,
                    etapa,
                    panel_sesion(nombre),
                    None
                )
                return

            mensaje = resto_mensaje

    # Si hay un TXT vinculado, el usuario puede consultarlo de inmediato
    # aunque todavía no haya registrado un nombre.
    if etapa == "nombre" and contexto_txt:
        etapa = "chat"
        if not nombre or nombre == "Invitad@":
            nombre = "Invitad@"

    # ETAPA CHAT
    historial.append({"rol": "user", "texto": mensaje, "hora": hora()})

    yield (
        render_topbar(nombre),
        render_chat(historial, nombre, typing=True),
        gr.update(value=""),
        historial,
        nombre,
        etapa,
        panel_sesion(nombre),
        None
    )

    time.sleep(0.15)

    inicio = time.time()
    if contexto_txt:
        respuesta = responder_sobre_archivo(mensaje, contexto_txt)
    else:
        respuesta = engine.responder(mensaje, nombre)
    tiempo_total = time.time() - inicio

    es_crisis = engine.detectar_crisis(mensaje)

    if es_crisis:
        texto_respuesta = respuesta
        respuesta_es_html = True
    else:
        recursos = engine.recursos_html(mensaje)
        respuesta_segura = html.escape(respuesta).replace("\n", "<br>")
        texto_respuesta = respuesta_segura

        if recursos:
            texto_respuesta += f"""
            <div class="inline-resources-divider"></div>
            {recursos}
            """

        respuesta_es_html = True

    historial.append({
        "rol": "bot",
        "texto": texto_respuesta,
        "hora": hora(),
        "html": respuesta_es_html
    })

    yield (
        render_topbar(nombre),
        render_chat(historial, nombre),
        gr.update(value="", placeholder="Escribe cómo te sientes hoy..."),
        historial,
        nombre,
        etapa,
        panel_analisis(historial, tiempo_total, "Texto"),
        gr.update(value=None, visible=False)
    )


def reiniciar():
    historial = estado_inicial()
    nombre = "Invitad@"
    etapa = "nombre"

    return (
        render_topbar(nombre),
        render_chat(historial, nombre),
        gr.update(value="", placeholder="Escribí tu nombre o contame cómo te sentís..."),
        historial,
        nombre,
        etapa,
        panel_inicial(),
        gr.update(value=None, visible=False)
    )




In [ ]:
# ==============================================================================
# 9. ESTILOS CSS ACTUALES


In [ ]:
# ==============================================================================
# CSS EXTERNO
# Los estilos se mantienen en GitHub para que el notebook sea más liviano.
# ==============================================================================

import requests

BASE_ASSETS_URL = "https://raw.githubusercontent.com/PaoRioColorado/EmotiChat/main"

ARCHIVOS_CSS = ["css/emotichat.css?v=27"]

def cargar_css_desde_github(ruta):
    url = f"{BASE_ASSETS_URL}/{ruta}"
    try:
        respuesta = requests.get(url, timeout=5)
        respuesta.raise_for_status()
        contenido = respuesta.text.strip()

        if not contenido:
            raise RuntimeError("El archivo está vacío.")

        print(f"CSS cargado: {ruta}")
        return contenido

    except Exception as error:
        print(f"Aviso: no se pudo cargar {ruta}: {error}")
        return ""

CSS = cargar_css_desde_github(ARCHIVOS_CSS[0])

if not CSS.strip():
    CSS = r"""
    :root { --primary: #6657c8; --text: #202941; --muted: #5f687c; --border: #dfe3ee; }
    .gradio-container { color: var(--text); background: #f7f8fc; }
    #app { width: 100%; max-width: 1500px; margin: 0 auto; }
    button { border-radius: 12px; }
    """
    print("Se utilizaron los estilos locales de respaldo.")


# Ajustes v7 incorporados directamente para no depender de cambios en GitHub.


# Ajustes finales v8: prioridad móvil y presentación académica.


# ==============================================================================
# V12 — CORRECCIÓN GLOBAL DE ANCHO, CENTRADO Y TARJETAS
# ==============================================================================


# ==============================================================================
# V13 — AJUSTE PUNTUAL DEL COMPOSITOR Y BOTONES DESHABILITADOS
# ==============================================================================


# ==============================================================================
# ESTILO FINAL DE INTERFAZ — V17
# Un único bloque final para layout, chat, avatar, botón y responsive.
# ==============================================================================


# ==============================================================================
# AJUSTES VISUALES V18
# ==============================================================================


# ==============================================================================
# AJUSTES V19 — CHAT SIN RECORTE HORIZONTAL
# Este bloque queda al final para reemplazar de forma controlada reglas heredadas.
# ==============================================================================


# Ajuste v22: contraste reforzado en la tarjeta de conclusión general.


# ==============================================================================
# AJUSTE DE VISIBILIDAD PARA LOS MÓDULOS ACADÉMICOS
# ==============================================================================


# ==============================================================================
# AJUSTES LOCALES V23 — MÓDULO TXT
# Estos estilos tienen prioridad sobre los CSS externos y corrigen el bloque
# que se veía desalineado o excesivamente alto.
# ==============================================================================


# ==============================================================================
# V24 — VISIBILIDAD REAL DEL MÓDULO TXT
# El módulo dejó de estar dentro de un acordeón y la tarjeta principal crece
# según su contenido. Evita que Gradio recorte los botones y resultados.
# ==============================================================================


# ==============================================================================
# V25 — MÓDULO TXT SIN RECORTES Y CON ACCIONES SIEMPRE VISIBLES
# ==============================================================================


# ==============================================================================
# V26 — TXT INTEGRADO, CONTRASTE Y CONTROLES PROFESIONALES
# ==============================================================================


# ==============================================================================
# AJUSTES FINALES — MÓDULO TXT Y VISTA RESPONSIVE
# ==============================================================================


# ===== Ajustes finales de contraste y versión móvil =====


In [ ]:
# ==============================================================================
# 10. CONSTRUCCIÓN Y EVENTOS DE LA APLICACIÓN


In [ ]:
# Ajustes finales del selector y del módulo TXT.

# Los audios no se descargan al iniciar la aplicación.
# Esto evita que una demora de Google Drive o GitHub bloquee la apertura de Gradio.
AUDIO_INFORMATIVO_LOCAL = None
AUDIO_EMOCIONAL_LOCAL = None
ESTADO_AUDIOS_PRUEBA = (
    "ℹ La aplicación está lista. Presioná ‘Cargar audios de prueba’ "
    "para descargar y habilitar los reproductores."
)

def cargar_audios_prueba_interfaz():
    """Descarga los WAV bajo demanda y actualiza los reproductores."""
    try:
        rutas = engine.descargar_audios_prueba()
        informativo = rutas.get("Informativo")
        emocional = rutas.get("Emocional")
        if not informativo or not emocional:
            raise RuntimeError("No se obtuvieron las dos rutas de audio.")
        return (
            gr.update(value=informativo),
            gr.update(value=emocional),
            " Audios cargados correctamente. Ya podés reproducirlos o ejecutar la comparación.",
        )
    except Exception as error:
        return (
            gr.update(value=None),
            gr.update(value=None),
            " No se pudieron cargar los audios. Revisá que los enlaces de Drive sean públicos. "
            f"Detalle: {error}",
        )


# Ajustes finales de interfaz y versión móvil.


# Correcciones finales de interfaz: prevalecen sobre estilos históricos.

with gr.Blocks(
    title=APP_NAME
) as app:

    JS_BASE_URL = "https://raw.githubusercontent.com/PaoRioColorado/EmotiChat/main/js"

    def cargar_javascript(nombre_archivo):
        respuesta = requests.get(
            f"{JS_BASE_URL}/{nombre_archivo}",
            timeout=5
        )
        respuesta.raise_for_status()

        contenido = respuesta.text
        if not contenido.strip():
            raise RuntimeError(
                f"El archivo JavaScript '{nombre_archivo}' está vacío."
            )

        return contenido

    try:
        with ThreadPoolExecutor(max_workers=2) as executor:
            AUTOSCROLL_JS, SCROLL_PANEL_JS = executor.map(
                cargar_javascript,
                ["autoscroll.js", "app.js"]
            )
    except Exception as error_js:
        print(f"Aviso: JavaScript externo no disponible: {error_js}")
        AUTOSCROLL_JS = "() => {}"
        SCROLL_PANEL_JS = "() => {}"


    historial_estado = gr.State(estado_inicial())
    nombre_estado = gr.State("Invitad@")
    etapa_estado = gr.State("nombre")
    audio_pendiente_estado = gr.State(None)
    ultimo_audio_estado = gr.State("")
    ultima_ruta_audio_estado = gr.State(None)
    contexto_txt_estado = gr.State("")
    nombre_txt_estado = gr.State("")
    contenido_txt_estado = gr.State("")

    with gr.Row(elem_id="app"):

        with gr.Column(elem_classes="sidebar"):
            gr.HTML(f"""
            <div class="brand">
                {logo_html("logo-img")}
                <div>
                    <h1>EmotiChat</h1>
                    <p>Asistente conversacional de bienestar emocional</p>
                </div>
            </div>
            """)

            with gr.Column(elem_classes="menu"):
                btn_menu_conversacion = gr.Button("Conversación", elem_classes=["side-button", "active"])
                btn_menu_audio = gr.Button("Procesamiento de voz", elem_classes="side-button")
                btn_menu_traduccion = gr.Button("Traducción", elem_classes="side-button")
                btn_menu_sentimientos = gr.Button("Sentimientos", elem_classes="side-button")
                btn_menu_conclusiones = gr.Button("Conclusiones", elem_classes="side-button")

            sidebar_detail = gr.HTML(
                sidebar_info("Conversación", "El chat está esperando que ingreses el nombre para iniciar la sesión."),
                elem_classes="sidebar-detail"
            )

            gr.HTML("""
            <div class="breathe-card">
                <div class="breathe-illustration">▰</div>
                <h3>Tecnologías integradas</h3>
                <p>Whisper, SpeechRecognition, traducción, análisis de sentimientos y síntesis de voz.</p>
            </div>
            """)

        with gr.Column(elem_classes="main"):
            topbar = gr.HTML(render_topbar("Invitad@"))

            with gr.Column(elem_classes="chat-card"):
                with gr.Row(elem_classes="chat-toolbar"):
                    reiniciar_btn = gr.Button(
                        "Nueva",
                        elem_id="reset",
                        elem_classes="compact-reset-btn",
                        scale=0,
                        min_width=96
                    )

                chat = gr.HTML(render_chat(estado_inicial()), elem_classes="chat-scroll", elem_id="chat_area")

                contexto_txt_indicador = gr.HTML(
                    '<div class="file-context-empty">Ningún archivo está vinculado al chat.</div>',
                    elem_id="chat_context_indicator"
                )

                with gr.Column(elem_classes="composer"):

                    with gr.Tabs(elem_classes="input-mode-tabs"):

                        with gr.Tab("Escribir", elem_id="tab_escribir"):

                            with gr.Row(elem_classes="message-composer-row"):
                                mensaje = gr.Textbox(
                                    placeholder="¿Cómo querés que te llame?",
                                    show_label=False,
                                    lines=1,
                                    max_lines=4,
                                    scale=1,
                                    min_width=0,
                                    container=True,
                                    elem_id="msg",
                                    elem_classes="message-input"
                                )

                                enviar = gr.Button(
                                    "Enviar",
                                    scale=0,
                                    min_width=88,
                                    elem_id="send",
                                    elem_classes=["send-button"],
                                    variant="primary"
                                )


                        with gr.Tab("Hablar", elem_id="tab_hablar"):

                            gr.HTML("""
                            <div class="voice-guide">
                                <strong>Enviar un mensaje de voz</strong>
                                <span>Grabá o elegí un audio. EmotiChat lo procesa automáticamente.</span>
                            </div>
                            """)

                            gr.HTML(
                                '<div class="voice-step">Grabar con el micrófono</div>'
                            )

                            audio_entrada = gr.Audio(
                                sources=["microphone"],
                                type="filepath",
                                format="wav",
                                label="Mensaje de voz",
                                elem_id="audio_input",
                                elem_classes="voice-recorder",
                                interactive=True
                            )

                            estado_audio = gr.HTML(
                                value="",
                                elem_id="voice_processing_status",
                                elem_classes="voice-processing-status"
                            )

                            gr.HTML(
                                '<div class="voice-step">Elegir un archivo de audio</div>'
                            )

                            archivo_audio = gr.UploadButton(
                                "Subir audio",
                                file_types=["audio"],
                                file_count="single",
                                type="filepath",
                                elem_id="upload_audio",
                                interactive=True
                            )



                    audio_out = gr.Audio(
                        label="Respuesta de EmotiChat",
                        type="filepath",
                        visible=False,
                        autoplay=True,
                        elem_id="audio_out"
                    )

                    audio_pipeline_normal = gr.Audio(
                        label="Resumen hablado - velocidad normal",
                        type="filepath",
                        visible=False,
                        autoplay=False
                    )

                    audio_pipeline_lento = gr.Audio(
                        label="Resumen hablado - velocidad lenta",
                        type="filepath",
                        visible=False,
                        autoplay=False
                    )

                    with gr.Group(elem_classes=["txt-section-card"]):
                        gr.HTML("""
                        <div class="txt-v26-header">
                            <h3>Trabajar con un archivo de texto</h3>
                            <p>Seleccioná un TXT. EmotiChat genera el audio automáticamente.</p>
                        </div>
                        """)

                        archivo_txt_tts = gr.UploadButton(
                            "Subir archivo TXT",
                            file_types=[".txt"],
                            file_count="single",
                            type="filepath",
                            elem_id="txt_file_upload",
                            elem_classes=["txt-file-input"]
                        )

                        estado_txt_tts = gr.Markdown(
                            "Esperando un archivo .txt.",
                            elem_classes=["txt-status"]
                        )

                        gr.HTML("<div class='txt-actions-label'><strong>Opciones del archivo</strong></div>")
                        with gr.Row(elem_classes=["txt-actions-v28"]):
                            analizar_txt_btn = gr.Button(
                                "Analizar sentimiento",
                                variant="secondary",
                                elem_id="txt_action_analysis"
                            )
                            conversar_txt_btn = gr.Button(
                                "Usar en la conversación",
                                variant="secondary",
                                elem_id="txt_action_context"
                            )
                            limpiar_txt_btn = gr.Button(
                                "Quitar archivo",
                                variant="secondary",
                                elem_id="txt_action_clear"
                            )

                        with gr.Column(visible=False, elem_id="txt_voice_section", elem_classes=["txt-control-card"]) as txt_voice_section:
                            gr.HTML("<h4>Configuración de voz</h4>")
                            with gr.Row(elem_classes=["txt-options-row"]):
                                idioma_txt_tts = gr.Dropdown(
                                    choices=["Español", "Inglés", "Francés", "Portugués", "Alemán"],
                                    value="Español",
                                    label="Idioma de lectura"
                                )
                                velocidad_txt_tts = gr.Radio(
                                    choices=["Lenta", "Normal", "Rápida"],
                                    value="Normal",
                                    label="Velocidad"
                                )

                            audio_txt_tts = gr.Audio(
                                label="Audio",
                                type="filepath",
                                visible=False,
                                autoplay=False,
                                elem_id="txt_audio_player",
                                elem_classes=["txt-audio-output"]
                            )

                        with gr.Accordion("Ver contenido del archivo", open=False, elem_classes=["txt-preview-accordion"]):
                            contenido_txt = gr.Textbox(
                                label="Vista previa",
                                placeholder="El contenido aparecerá acá.",
                                lines=6,
                                interactive=False,
                                elem_classes=["txt-preview"]
                            )

                        with gr.Column(visible=False, elem_id="txt_analysis_section", elem_classes=["txt-analysis-shell"]) as txt_analysis_section:
                            resultado_analisis_txt = gr.Markdown(
                                elem_classes=["txt-analysis-result"]
                            )

                        with gr.Column(visible=False, elem_id="txt_context_result_section", elem_classes=["txt-context-result-shell"]) as txt_context_result_section:
                            resultado_contexto_txt = gr.Markdown(
                                elem_classes=["txt-context-result"]
                            )


            panel = gr.HTML(panel_inicial(), elem_classes="module-panel")

            with gr.Column(
                visible=False,
                elem_id="audio_tools_section",
                elem_classes=["audio-tools-section"]
            ) as audio_tools_section:
                gr.HTML("""
                <div class="panel-title audio-panel-title">
                    <span class="section-mark" aria-hidden="true"></span>
                    <div>
                        <h2>Procesamiento de voz</h2>
                        <p>Grabación, transcripción, análisis y comparación de audio</p>
                    </div>
                </div>
                """)

                with gr.Tabs(elem_id="audio_tools_tabs"):
                    with gr.Tab("Análisis completo"):
                        with gr.Column(elem_classes=["audio-workspace-card"]):
                            gr.HTML("""
                            <div class="voice-pipeline-heading">
                                <h2>Pipeline completo de audio</h2>
                                <p>Transcribe el audio, analiza el sentimiento, genera un resumen y crea una versión hablada.</p>
                            </div>
                            """)

                            gr.Markdown(
                                "**Opción 1 — Audio del chat**\n\n"
                                "Usa el último mensaje grabado o subido en Conversación.",
                                elem_classes=["audio-option-help"]
                            )
                            analizar_audio_chat_btn = gr.Button(
                                "Analizar último audio del chat",
                                variant="primary",
                                elem_id="analyze_chat_audio_button"
                            )

                            gr.HTML('<div class="audio-choice-divider"><span>o</span></div>')

                            gr.Markdown(
                                "**Opción 2 — Otro archivo de audio**\n\n"
                                "Seleccioná un archivo guardado en el teléfono o la computadora.",
                                elem_classes=["audio-option-help"]
                            )
                            pipeline_audio_entrada = gr.Audio(
                                sources=["upload"],
                                type="filepath",
                                label="Audio seleccionado para el Pipeline",
                                interactive=True,
                                buttons=[],
                                elem_id="pipeline_manual_audio"
                            )
                            analizar_audio_archivo_btn = gr.Button(
                                "Analizar archivo seleccionado",
                                variant="secondary",
                                elem_id="analyze_selected_audio_button"
                            )

                            pipeline_transcripcion = gr.Textbox(
                                label="1. Transcripción",
                                lines=4,
                                interactive=False,
                                elem_id="pipeline_transcription"
                            )
                            pipeline_sentimiento = gr.Textbox(
                                label="2. Sentimiento predominante",
                                interactive=False,
                                elem_id="pipeline_sentiment"
                            )
                            pipeline_resumen = gr.Textbox(
                                label="3. Resumen generado",
                                lines=5,
                                interactive=False,
                                elem_id="pipeline_summary"
                            )
                            pipeline_audio_final = gr.Audio(
                                label="4. Audio del resumen",
                                type="filepath",
                                visible=False,
                                buttons=[],
                                elem_id="pipeline_summary_audio"
                            )
                            estado_pipeline_audio = gr.Markdown()

                    with gr.Tab("Comparar transcripciones"):
                        with gr.Column(elem_classes=["audio-workspace-card"]):
                            gr.HTML("""
                            <div class="comparison-heading">
                                <span class="comparison-kicker">Comparación técnica</span>
                                <h2>Whisper y SpeechRecognition</h2>
                                <p>Compará las transcripciones usando los WAV de prueba o un audio propio.</p>
                            </div>
                            """)

                            with gr.Column(elem_classes=["academic-audio-shell"]):
                                gr.HTML("""
                                <div class="academic-audio-header">
                                    <h2>Audios WAV de prueba</h2>
                                    <p>Primero prepará los audios; después podés reproducirlos y ejecutar la prueba.</p>
                                </div>
                                """)

                                cargar_audios_prueba_btn = gr.Button(
                                    "Preparar audios WAV",
                                    variant="primary",
                                    elem_id="load_project_audios_button"
                                )

                                with gr.Row(elem_classes=["project-audio-grid"]):
                                    with gr.Column(elem_classes=["academic-audio-card"]):
                                        gr.Markdown("### Audio informativo")
                                        audio_informativo_proyecto = gr.Audio(
                                            value=AUDIO_INFORMATIVO_LOCAL,
                                            label="Audio informativo · WAV · 14 segundos",
                                            type="filepath",
                                            interactive=False,
                                            buttons=[],
                                            elem_id="project_audio_informative"
                                        )

                                    with gr.Column(elem_classes=["academic-audio-card"]):
                                        gr.Markdown("### Audio emocional")
                                        audio_emocional_proyecto = gr.Audio(
                                            value=AUDIO_EMOCIONAL_LOCAL,
                                            label="Audio emocional · WAV · 15 segundos",
                                            type="filepath",
                                            interactive=False,
                                            buttons=[],
                                            elem_id="project_audio_emotional"
                                        )

                                estado_audios_proyecto = gr.Markdown(
                                    ESTADO_AUDIOS_PRUEBA,
                                    elem_id="project_audio_status"
                                )

                                ejecutar_audios_proyecto_btn = gr.Button(
                                    "Ejecutar prueba con los 2 WAV",
                                    variant="secondary",
                                    elem_id="run_project_audios_button"
                                )
                                resultado_audios_proyecto = gr.HTML(
                                    value="",
                                    elem_id="project_audios_result"
                                )

                            with gr.Column(elem_classes=["own-audio-comparison"]):
                                gr.HTML("""
                                <div class="own-audio-heading">
                                    <h3>Comparar un audio propio</h3>
                                    <p>Seleccioná un audio y escribí exactamente lo que se escucha.</p>
                                </div>
                                """)

                                with gr.Row(elem_classes=["comparison-audio-sources"]):
                                    with gr.Column(elem_classes=["comparison-source-card"]):
                                        gr.Markdown("**Grabar un audio**")
                                        audio_comparacion_grabado = gr.Audio(
                                            sources=["microphone"],
                                            type="filepath",
                                            format="wav",
                                            label="Grabación para comparar",
                                            interactive=True,
                                            buttons=[],
                                            elem_id="comparison_recorded_audio"
                                        )

                                    with gr.Column(elem_classes=["comparison-source-card"]):
                                        gr.Markdown("**Seleccionar un archivo**")
                                        audio_comparacion_archivo = gr.Audio(
                                            sources=["upload"],
                                            type="filepath",
                                            label="Archivo para comparar",
                                            interactive=True,
                                            buttons=[],
                                            elem_id="comparison_uploaded_audio"
                                        )

                                texto_referencia_transcripcion = gr.Textbox(
                                    label="Texto de referencia",
                                    placeholder="Escribí exactamente lo que se escucha en el audio...",
                                    lines=3,
                                    max_lines=6,
                                    elem_id="transcription_reference_text"
                                )

                                comparar_transcripcion_btn = gr.Button(
                                    "Comparar audio y texto",
                                    variant="primary",
                                    elem_id="compare_transcriptions_button"
                                )

                                resultado_comparacion_transcripcion = gr.HTML(
                                    value="",
                                    elem_id="transcription_comparison_result"
                                )

                    with gr.Tab("Texto a voz"):
                        with gr.Column(scale=0, elem_classes=["tts-experiment-card"]):
                            gr.HTML("""
                            <div class="tts-experiment-heading">
                                <span class="audio-tools-kicker">Síntesis de voz</span>
                                <h2>Comparar voces, idiomas y velocidades</h2>
                                <p>Seleccioná un TXT, generá distintas versiones y escuchá las diferencias.</p>
                            </div>
                            """)

                            archivo_tts_experimento = gr.UploadButton(
                                "Seleccionar archivo TXT",
                                file_types=[".txt"],
                                file_count="single",
                                type="filepath",
                                elem_id="tts_experiment_upload"
                            )

                            with gr.Row(elem_classes=["tts-experiment-options"]):
                                idioma_tts_experimento = gr.Dropdown(
                                    choices=["Español", "Inglés", "Francés", "Portugués", "Alemán"],
                                    value="Español",
                                    label="Idioma de la voz",
                                    elem_id="tts_experiment_language"
                                )
                                velocidad_tts_experimento = gr.Radio(
                                    choices=["Lenta", "Normal", "Rápida"],
                                    value="Normal",
                                    label="Velocidad",
                                    elem_id="tts_experiment_speed"
                                )

                            estado_tts_experimento = gr.Markdown(
                                "Seleccioná un TXT para comenzar la comparación.",
                                elem_classes=["tts-experiment-status"]
                            )
                            audio_tts_experimento = gr.Audio(
                                label="Versión de voz generada",
                                type="filepath",
                                visible=False,
                                buttons=[],
                                elem_id="tts_experiment_audio"
                            )
                            texto_tts_experimento = gr.State("")

                            gr.Markdown(
                                "**Cómo comparar la calidad:** escuchá el mismo texto en varios idiomas y velocidades. "
                                "Observá pronunciación, claridad, naturalidad, ritmo y pausas.",
                                elem_classes=["tts-comparison-guide"]
                            )

            with gr.Row(elem_classes="pdf-actions", visible=False) as acciones_pdf:
                btn_generar_pdf = gr.Button(
                    "Generar PDF final",
                    visible=True,
                    variant="primary",
                    elem_id="generate_pdf_button"
                )
                pdf_out = gr.DownloadButton(
                    "Descargar informe generado",
                    visible=False,
                    variant="secondary",
                    elem_id="download_pdf_button"
                )

    FOOTER_URL = "https://raw.githubusercontent.com/PaoRioColorado/EmotiChat/main/html/footer.html"
    try:
        respuesta_footer = requests.get(FOOTER_URL, timeout=3)
        respuesta_footer.raise_for_status()
        FOOTER_HTML = respuesta_footer.text
    except Exception:
        FOOTER_HTML = (
            '<footer style="text-align:center;padding:18px;color:#586078">'
            'EmotiChat v2.0 · Developed by Paola F. Dueña'
            '</footer>'
        )
    gr.HTML(FOOTER_HTML)

    app.load(
        fn=None,
        inputs=None,
        outputs=None,
        js=AUTOSCROLL_JS
    )

    MENU_BUTTONS_JS = [
        "#app .side-button button",
        "#app button.side-button"
    ]

    def menu_js(indice):
        return f"""() => {{
          const botones = Array.from(document.querySelectorAll('#app .side-button button, #app button.side-button'));
          botones.forEach((boton, posicion) => boton.classList.toggle('menu-selected', posicion === {indice}));
        }}"""

    btn_menu_conversacion.click(
        mostrar_conversacion,
        inputs=[nombre_estado, etapa_estado],
        outputs=[sidebar_detail, panel],
        js=menu_js(0)
    )

    evento_menu_audio = btn_menu_audio.click(
        mostrar_audio,
        inputs=[historial_estado],
        outputs=[sidebar_detail, panel],
        js=menu_js(1)
    )

    evento_menu_audio.then(
        lambda: gr.update(visible=True),
        inputs=None,
        outputs=[audio_tools_section]
    ).then(
        fn=None,
        inputs=None,
        outputs=None,
        js="""() => setTimeout(() => {
          document.getElementById('audio_tools_section')?.scrollIntoView({
            behavior: 'smooth',
            block: 'start'
          });
        }, 180)"""
    )

    btn_menu_traduccion.click(
        mostrar_traduccion,
        inputs=[historial_estado],
        outputs=[sidebar_detail, panel],
        js=menu_js(2)
    ).then(fn=None, inputs=None, outputs=None, js=SCROLL_PANEL_JS)

    btn_menu_sentimientos.click(
        mostrar_sentimientos,
        inputs=[historial_estado],
        outputs=[sidebar_detail, panel],
        js=menu_js(3)
    ).then(fn=None, inputs=None, outputs=None, js=SCROLL_PANEL_JS)

    btn_menu_conclusiones.click(
        mostrar_conclusiones,
        inputs=[historial_estado],
        outputs=[sidebar_detail, panel, acciones_pdf],
        js=menu_js(4)
    ).then(fn=None, inputs=None, outputs=None, js=SCROLL_PANEL_JS)

    # El informe se ofrece únicamente al abrir Conclusiones.
    for boton_modulo in [btn_menu_conversacion, btn_menu_audio, btn_menu_traduccion, btn_menu_sentimientos]:
        boton_modulo.click(
            lambda: gr.update(visible=False),
            inputs=None,
            outputs=[acciones_pdf]
        )


    for boton_sin_audio in [
        btn_menu_conversacion,
        btn_menu_traduccion,
        btn_menu_sentimientos,
        btn_menu_conclusiones
    ]:
        boton_sin_audio.click(
            lambda: gr.update(visible=False),
            inputs=None,
            outputs=[audio_tools_section]
        )

    cargar_audios_prueba_btn.click(
        fn=cargar_audios_prueba_interfaz,
        inputs=None,
        outputs=[
            audio_informativo_proyecto,
            audio_emocional_proyecto,
            estado_audios_proyecto,
        ],
        show_progress="full",
    )

    ejecutar_audios_proyecto_btn.click(
        ejecutar_pruebas_audios_proyecto,
        inputs=None,
        outputs=[resultado_audios_proyecto],
        show_progress="full"
    )

    comparar_transcripcion_btn.click(
        comparar_audio_con_referencia,
        inputs=[
            audio_comparacion_grabado,
            audio_comparacion_archivo,
            texto_referencia_transcripcion
        ],
        outputs=[resultado_comparacion_transcripcion],
        show_progress="full"
    )


    # Al seleccionar un TXT se lee el contenido, se genera el audio y aparece el reproductor.
    archivo_txt_tts.change(
        fn=procesar_txt_al_cargar,
        inputs=[archivo_txt_tts, idioma_txt_tts, velocidad_txt_tts],
        outputs=[
            contenido_txt_estado,
            contenido_txt,
            estado_txt_tts,
            nombre_txt_estado,
            audio_txt_tts,
            txt_voice_section
        ],
        show_progress="full",
        trigger_mode="always_last"
    )

    # Una vez generado, cambiar idioma o velocidad actualiza el audio automáticamente.
    velocidad_txt_tts.change(
        fn=convertir_texto_cargado_a_audio,
        inputs=[contenido_txt_estado, archivo_txt_tts, idioma_txt_tts, velocidad_txt_tts],
        outputs=[audio_txt_tts, estado_txt_tts, txt_voice_section],
        show_progress="hidden"
    )
    idioma_txt_tts.change(
        fn=convertir_texto_cargado_a_audio,
        inputs=[contenido_txt_estado, archivo_txt_tts, idioma_txt_tts, velocidad_txt_tts],
        outputs=[audio_txt_tts, estado_txt_tts, txt_voice_section],
        show_progress="hidden"
    )

    # Experimento visible de síntesis: mismo TXT, distintos idiomas y velocidades.
    archivo_tts_experimento.upload(
        fn=preparar_experimento_tts,
        inputs=[archivo_tts_experimento, idioma_tts_experimento, velocidad_tts_experimento],
        outputs=[texto_tts_experimento, audio_tts_experimento, estado_tts_experimento],
        show_progress="full"
    )
    idioma_tts_experimento.change(
        fn=actualizar_experimento_tts,
        inputs=[texto_tts_experimento, archivo_tts_experimento, idioma_tts_experimento, velocidad_tts_experimento],
        outputs=[audio_tts_experimento, estado_tts_experimento],
        show_progress="hidden"
    )
    velocidad_tts_experimento.change(
        fn=actualizar_experimento_tts,
        inputs=[texto_tts_experimento, archivo_tts_experimento, idioma_tts_experimento, velocidad_tts_experimento],
        outputs=[audio_tts_experimento, estado_tts_experimento],
        show_progress="hidden"
    )

    analizar_txt_evento = analizar_txt_btn.click(
        fn=lambda texto, archivo: (analizar_texto_txt(texto, archivo), gr.update(visible=True)),
        inputs=[contenido_txt_estado, archivo_txt_tts],
        outputs=[resultado_analisis_txt, txt_analysis_section],
        show_progress="full",
        js="""() => {
          const ids = ['txt_action_analysis','txt_action_context','txt_action_clear'];
          ids.forEach(id => document.querySelector('#'+id+' button')?.classList.remove('txt-action-selected'));
          document.querySelector('#txt_action_analysis button')?.classList.add('txt-action-selected');
        }"""
    )
    analizar_txt_evento.then(
        fn=None, inputs=None, outputs=None,
        js="""() => setTimeout(() => {
          document.getElementById('txt_analysis_section')?.scrollIntoView({behavior:'smooth', block:'start'});
        }, 120)"""
    )

    conversar_txt_evento = conversar_txt_btn.click(
        fn=lambda texto, archivo, nombre_archivo, historial, nombre, etapa: (*activar_contexto_txt(texto, archivo, nombre_archivo, historial, nombre, etapa), gr.update(visible=False)),
        inputs=[contenido_txt_estado, archivo_txt_tts, nombre_txt_estado, historial_estado, nombre_estado, etapa_estado],
        outputs=[
            contexto_txt_estado, nombre_txt_estado, historial_estado,
            chat, contexto_txt_indicador, resultado_contexto_txt,
            txt_context_result_section
        ],
        show_progress="hidden",
        js="""() => {
          const ids = ['txt_action_analysis','txt_action_context','txt_action_clear'];
          ids.forEach(id => document.querySelector('#'+id+' button')?.classList.remove('txt-action-selected'));
          document.querySelector('#txt_action_context button')?.classList.add('txt-action-selected');
        }"""
    )
    conversar_txt_evento.then(
        fn=None, inputs=None, outputs=None,
        js="""() => setTimeout(() => {
          document.getElementById('chat_area')?.scrollIntoView({behavior:'smooth', block:'start'});
          document.getElementById('msg')?.querySelector('textarea')?.focus();
        }, 120)"""
    )

    limpiar_txt_btn.click(
        fn=lambda: (*limpiar_modulo_txt(), gr.update(visible=False), gr.update(visible=False), gr.update(visible=False)),
        inputs=[],
        outputs=[
            archivo_txt_tts, contenido_txt_estado, contenido_txt, estado_txt_tts, audio_txt_tts,
            resultado_analisis_txt, contexto_txt_estado, nombre_txt_estado,
            contexto_txt_indicador, resultado_contexto_txt,
            txt_voice_section, txt_analysis_section, txt_context_result_section
        ],
        show_progress="hidden",
        js="() => {\n  const ids = ['txt_action_analysis','txt_action_context','txt_action_clear'];\n  ids.forEach(id => document.querySelector('#'+id+' button')?.classList.remove('txt-action-selected'));\n  const btn = document.querySelector('#txt_action_clear button'); if (btn) btn.classList.add('txt-action-selected');\n  setTimeout(() => btn?.classList.remove('txt-action-selected'), 500);\n}"
    )

    analizar_audio_chat_btn.click(
        fn=ejecutar_pipeline_desde_audio,
        inputs=[ultima_ruta_audio_estado],
        outputs=[
            pipeline_transcripcion,
            pipeline_sentimiento,
            pipeline_resumen,
            pipeline_audio_final,
            estado_pipeline_audio
        ],
        show_progress="full"
    )

    analizar_audio_archivo_btn.click(
        fn=ejecutar_pipeline_desde_audio,
        inputs=[pipeline_audio_entrada],
        outputs=[
            pipeline_transcripcion,
            pipeline_sentimiento,
            pipeline_resumen,
            pipeline_audio_final,
            estado_pipeline_audio
        ],
        show_progress="full"
    )


    # El archivo elegido se confirma y procesa automáticamente.
    archivo_audio.upload(
        lambda archivo: (
            '<div class="processing-card">'
            '<span class="processing-spinner"></span>'
            '<div><strong>Audio cargado</strong>'
            '<span>Procesando el mensaje...</span></div></div>'
        ) if archivo else '',
        inputs=[archivo_audio],
        outputs=[estado_audio],
        show_progress="hidden"
    ).then(
        procesar_modo_voz,
        inputs=[
            archivo_audio,
            historial_estado,
            nombre_estado,
            etapa_estado,
            ultimo_audio_estado,
            ultima_ruta_audio_estado
        ],
        outputs=[
            topbar,
            chat,
            mensaje,
            historial_estado,
            nombre_estado,
            etapa_estado,
            panel,
            audio_out,
            archivo_audio,
            ultimo_audio_estado,
            ultima_ruta_audio_estado
        ],
        show_progress="hidden"
    ).then(
        lambda: (
            '<div class="processing-card processing-success">'
            '<div><strong>Audio procesado</strong>'
            '<span>El mensaje se agregó a la conversación.</span></div></div>'
        ),
        inputs=None,
        outputs=[estado_audio],
        show_progress="hidden"
    )

    # Flujo automático de la grabación:
    # muestra un aviso, espera el archivo, procesa y luego limpia el aviso.
    audio_entrada.change(
        lambda: """
        <div class="processing-card">
            <span class="processing-spinner"></span>
            <div>
                <strong>Procesando audio...</strong>
                <span>EmotiChat está transcribiendo tu mensaje. Puede tardar unos segundos.</span>
            </div>
        </div>
        """,
        inputs=None,
        outputs=[estado_audio],
        show_progress="hidden"
    ).then(
        esperar_audio_guardado,
        inputs=[audio_entrada],
        outputs=[audio_pendiente_estado],
        show_progress="hidden"
    ).then(
        procesar_modo_voz,
        inputs=[
            audio_pendiente_estado,
            historial_estado,
            nombre_estado,
            etapa_estado,
            ultimo_audio_estado,
            ultima_ruta_audio_estado
        ],
        outputs=[
            topbar,
            chat,
            mensaje,
            historial_estado,
            nombre_estado,
            etapa_estado,
            panel,
            audio_out,
            audio_entrada,
            ultimo_audio_estado,
            ultima_ruta_audio_estado
        ],
        show_progress="hidden"
    ).then(
        lambda: "",
        inputs=None,
        outputs=[estado_audio],
        show_progress="hidden"
    ).then(
        lambda: None,
        inputs=None,
        outputs=[audio_pendiente_estado],
        show_progress="hidden"
    )

    btn_generar_pdf.click(
        generar_pdf_descargable,
        inputs=[historial_estado, nombre_estado],
        outputs=[pdf_out]
    )

    enviar.click(
        procesar,
        inputs=[mensaje, historial_estado, nombre_estado, etapa_estado, contexto_txt_estado, nombre_txt_estado],
        outputs=[topbar, chat, mensaje, historial_estado, nombre_estado, etapa_estado, panel, audio_out]
    )

    mensaje.submit(
        procesar,
        inputs=[mensaje, historial_estado, nombre_estado, etapa_estado, contexto_txt_estado, nombre_txt_estado],
        outputs=[topbar, chat, mensaje, historial_estado, nombre_estado, etapa_estado, panel, audio_out]
    )

    reiniciar_evento = reiniciar_btn.click(
        reiniciar,
        inputs=[],
        outputs=[
            topbar,
            chat,
            mensaje,
            historial_estado,
            nombre_estado,
            etapa_estado,
            panel,
            audio_out
        ]
    )

    reiniciar_evento.then(
        lambda: "",
        inputs=None,
        outputs=[ultimo_audio_estado],
        show_progress="hidden"
    )

    reiniciar_evento.then(
        lambda: ("", "", '<div class="file-context-empty">Ningún archivo está vinculado al chat.</div>'),
        inputs=None,
        outputs=[contexto_txt_estado, nombre_txt_estado, contexto_txt_indicador],
        show_progress="hidden"
    )


# Ajustes de presentación y equivalencia funcional entre escritorio y celular.


# Refinamiento visual final: iconografía consistente y acciones menos oscuras.

# Lanzamiento externo confiable para Google Colab.
# Se intenta abrir una pestaña y también se muestra un botón de respaldo.
from IPython.display import HTML as ColabHTML, display as colab_display

resultado_launch = app.queue().launch(
    share=True,
    debug=False,
    inline=False,
    inbrowser=False,
    prevent_thread_lock=True,
    css=CSS,
    theme=gr.themes.Soft(),
    show_error=True,
    quiet=False
)

# launch() puede devolver un objeto o una tupla según la versión de Gradio.
candidatos_url = [
    getattr(resultado_launch, "share_url", None),
    getattr(app, "share_url", None),
]
if isinstance(resultado_launch, (tuple, list)):
    candidatos_url.extend(resultado_launch)

gradio_url = next(
    (
        str(valor) for valor in candidatos_url
        if isinstance(valor, str)
        and valor.startswith("https://")
        and "gradio.live" in valor
    ),
    None
)

if gradio_url:
    colab_display(ColabHTML(f"""
    <div style="padding:24px;text-align:center;font-family:Arial,sans-serif">
      <h2 style="color:#5146a5;margin:0 0 14px">EmotiChat está listo</h2>
      <a href="{gradio_url}" target="_blank" rel="noopener noreferrer"
         style="display:inline-block;padding:15px 28px;border-radius:14px;background:#6657c8;color:white;text-decoration:none;font-size:18px;font-weight:700;box-shadow:0 8px 22px rgba(102,87,200,.28)">
         Abrir EmotiChat en Gradio
      </a>
      <p style="color:#62677a;margin-top:12px">Si Chrome bloquea la apertura automática, presioná el botón.</p>
      <script>setTimeout(() => window.open('{gradio_url}', '_blank', 'noopener'), 500);</script>
    </div>
    """))
else:
    print("No se obtuvo el enlace público. Volvé a ejecutar solamente esta celda.")
